In [1]:
print("Hello")

Hello


In [ ]:
# ============================================================
# 05_stage5_master_rgb_singlecell.ipynb
# ONE-CELL, RESUMABLE, STABLE STAGE-5 RGB BENCHMARK
# ============================================================

import os
import gc
import json
import time
import copy
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import (
    resnet18, ResNet18_Weights,
    efficientnet_b0, EfficientNet_B0_Weights,
    mobilenet_v3_small, MobileNet_V3_Small_Weights,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)

# ============================================================
# 1. SETTINGS
# ============================================================
SEED = 42

IMG_SIZE = 224
NUM_SAMPLE_FRAMES = 32

# 400 is not practical here. Use stable values.
BATCH_SIZE_BY_MODEL = {
    "resnet18": 24,
    "efficientnet_b0": 16,
    "mobilenet_v3_small": 32,
}

EPOCHS = 12
LR = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 4

USE_AMP = True
USE_PRETRAINED = True

# Jupyter-safe
NUM_WORKERS = 0
PIN_MEMORY = False

# Optional quick debugging
LIMIT_TRAIN = None
LIMIT_VAL = None
LIMIT_TEST = None

# Smart sampling from middle useful region
SAMPLE_START_FRAC = 0.10
SAMPLE_END_FRAC = 0.90

# Resume / skip finished runs automatically
SKIP_IF_METRICS_EXISTS = True

# Run all Stage 5 combinations
FRAME_SOURCES = [
    "RGB",
    "RGB_BGREM",
]

MODELS_TO_RUN = [
    "resnet18",
    "efficientnet_b0",
    "mobilenet_v3_small",
]

# protocol_name, train_csv, val_csv, test_csv
SPLIT_CONFIGS = [
    ("protocol_A",          "protocol_A_train.csv",          "protocol_A_val.csv",          "protocol_A_test.csv"),
    ("protocol_B_4_to_8",   "protocol_B_train_4_test_8.csv", None,                          "protocol_B_test_8.csv"),
    ("protocol_B_8_to_4",   "protocol_B_train_8_test_4.csv", None,                          "protocol_B_test_4.csv"),
    ("protocol_B_4_6_to_8", "protocol_B_train_4_6_test_8.csv", None,                        "protocol_B_test_8_from_4_6.csv"),
    ("protocol_B_4_8_to_6", "protocol_B_train_4_8_test_6.csv", None,                        "protocol_B_test_6_from_4_8.csv"),
    ("protocol_B_6_8_to_4", "protocol_B_train_6_8_test_4.csv", None,                        "protocol_B_test_4_from_6_8.csv"),
]

CLASS_NAMES = [
    "doctor",
    "emergency",
    "fire",
    "help",
    "robbery",
    "sit_down",
    "stand_up",
]
label2id = {c: i for i, c in enumerate(CLASS_NAMES)}
id2label = {i: c for c, i in label2id.items()}
NUM_CLASSES = len(CLASS_NAMES)

# ============================================================
# 2. REPRODUCIBILITY
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ============================================================
# 3. PATHS
# ============================================================
cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd

SPLIT_DIR = ROOT / "data" / "processed" / "splits"
MANIFEST_DIR = ROOT / "data" / "processed" / "manifests"

RESULTS_ROOT = ROOT / "results" / "stage5_rgb_master"
CHECKPOINT_ROOT = ROOT / "checkpoints" / "stage5_rgb_master"
LOG_ROOT = ROOT / "logs" / "stage5_rgb_master"

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
LOG_ROOT.mkdir(parents=True, exist_ok=True)

FRAME_SUMMARY_CSV = MANIFEST_DIR / "frame_extraction_trimmed_summary.csv"
if not FRAME_SUMMARY_CSV.exists():
    raise FileNotFoundError(f"Missing file: {FRAME_SUMMARY_CSV}")

print("ROOT:", ROOT)
print("FRAME_SUMMARY_CSV:", FRAME_SUMMARY_CSV)
print("CLASS_NAMES:", CLASS_NAMES)

# ============================================================
# 4. DEVICE
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
torch.backends.cudnn.benchmark = False

# ============================================================
# 5. FRAME SUMMARY
# ============================================================
frame_df_all = pd.read_csv(FRAME_SUMMARY_CSV)
frame_df_all = frame_df_all[frame_df_all["status"] == "success"].copy()
frame_df_all = frame_df_all[["pair_id", "modality", "output_dir", "frames_extracted"]].drop_duplicates()

# ============================================================
# 6. HELPERS
# ============================================================
def sample_indices_middle_region(n_total, n_samples, start_frac=0.10, end_frac=0.90):
    if n_total <= 0:
        return [0] * n_samples

    start_idx = int(round(start_frac * (n_total - 1)))
    end_idx = int(round(end_frac * (n_total - 1)))

    start_idx = max(0, min(start_idx, n_total - 1))
    end_idx = max(start_idx, min(end_idx, n_total - 1))

    idxs = np.linspace(start_idx, end_idx, n_samples)
    idxs = np.round(idxs).astype(int)
    return np.clip(idxs, 0, n_total - 1).tolist()

def frame_index_to_path(frame_dir, idx):
    return Path(frame_dir) / f"frame_{idx+1:06d}.jpg"

def load_split_df(csv_name):
    csv_path = SPLIT_DIR / csv_name
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing split file: {csv_path}")
    df = pd.read_csv(csv_path)
    if "is_valid_pair" in df.columns:
        df = df[df["is_valid_pair"] == True].copy()
    return df.reset_index(drop=True)

def attach_labels(df_):
    out = df_.copy()
    out = out[out["gesture"].isin(CLASS_NAMES)].copy()
    out["label"] = out["gesture"].map(label2id)
    return out.reset_index(drop=True)

def attach_frame_info(df_, frame_source):
    sub = frame_df_all[frame_df_all["modality"] == frame_source].copy()
    sub = sub.rename(columns={
        "output_dir": "frame_dir",
        "frames_extracted": "num_frames_available"
    })
    sub = sub[["pair_id", "frame_dir", "num_frames_available"]].drop_duplicates()

    out = df_.merge(sub, on="pair_id", how="left")
    out = out[out["num_frames_available"].fillna(0) > 0].copy().reset_index(drop=True)
    out["num_frames_available"] = out["num_frames_available"].astype(int)
    return out

# ============================================================
# 7. TRANSFORMS
# ============================================================
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

# ============================================================
# 8. DATASET
# ============================================================
class FrameVotingDataset(Dataset):
    def __init__(self, df, transform, num_sample_frames=32, start_frac=0.10, end_frac=0.90):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform
        self.sampled_paths = []

        for _, row in self.df.iterrows():
            idxs = sample_indices_middle_region(
                int(row["num_frames_available"]),
                num_sample_frames,
                start_frac,
                end_frac
            )
            self.sampled_paths.append([frame_index_to_path(row["frame_dir"], i) for i in idxs])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        paths = self.sampled_paths[idx]

        frames = []
        for p in paths:
            try:
                with Image.open(p) as f:
                    img = f.convert("RGB")
            except Exception:
                img = Image.fromarray(np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))
            frames.append(self.transform(img))

        frames = torch.stack(frames, dim=0)
        label = int(row["label"])

        meta = {
            "pair_id": row["pair_id"],
            "subject_id": row["subject_id"],
            "distance": row["distance"],
            "gesture": row["gesture"],
            "frame_dir": row["frame_dir"],
        }
        return frames, label, meta

def collate_fn(batch):
    frames = torch.stack([x[0] for x in batch], dim=0)
    labels = torch.tensor([x[1] for x in batch], dtype=torch.long)
    metas = [x[2] for x in batch]
    return frames, labels, metas

# ============================================================
# 9. MODEL FACTORY
# ============================================================
def create_model(model_name, num_classes, use_pretrained=True):
    if model_name == "resnet18":
        model = resnet18(weights=ResNet18_Weights.DEFAULT if use_pretrained else None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif model_name == "efficientnet_b0":
        model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT if use_pretrained else None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    elif model_name == "mobilenet_v3_small":
        model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT if use_pretrained else None)
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)

    else:
        raise ValueError(f"Unsupported model: {model_name}")

    return model

def forward_video_voting(model, frames):
    B, T, C, H, W = frames.shape
    frames = frames.view(B * T, C, H, W)
    frame_logits = model(frames)
    frame_logits = frame_logits.view(B, T, -1)
    video_logits = frame_logits.mean(dim=1)
    return video_logits

# ============================================================
# 10. TRAIN / EVAL
# ============================================================
def run_one_epoch(model, loader, criterion, optimizer=None, scaler=None, train=True):
    model.train() if train else model.eval()

    total_loss = 0.0
    all_labels = []
    all_preds = []

    for frames, labels, metas in loader:
        frames, labels = frames.to(device), labels.to(device)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(USE_AMP and device.type == "cuda")
            ):
                video_logits = forward_video_voting(model, frames)
                loss = criterion(video_logits, labels)

            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        preds = video_logits.argmax(dim=1)

        total_loss += loss.item() * labels.size(0)
        all_labels.extend(labels.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(all_labels, all_preds),
        "macro_f1": f1_score(all_labels, all_preds, average="macro"),
        "weighted_f1": f1_score(all_labels, all_preds, average="weighted"),
    }

@torch.no_grad()
def predict_with_metadata(model, loader):
    model.eval()

    all_labels = []
    all_preds = []
    rows = []

    start_time = time.time()
    total_samples = 0

    for frames, labels, metas in loader:
        frames, labels = frames.to(device), labels.to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(USE_AMP and device.type == "cuda")
        ):
            video_logits = forward_video_voting(model, frames)
            probs = torch.softmax(video_logits, dim=1)
            preds = probs.argmax(dim=1)

        probs_np = probs.detach().cpu().numpy()
        preds_np = preds.detach().cpu().numpy()
        labels_np = labels.detach().cpu().numpy()

        total_samples += len(labels_np)

        all_labels.extend(labels_np.tolist())
        all_preds.extend(preds_np.tolist())

        for i, meta in enumerate(metas):
            row = {
                "pair_id": meta["pair_id"],
                "subject_id": meta["subject_id"],
                "distance": meta["distance"],
                "gesture_true": id2label[int(labels_np[i])],
                "gesture_pred": id2label[int(preds_np[i])],
                "frame_dir": meta["frame_dir"],
            }
            for c_idx, c_name in id2label.items():
                row[f"prob_{c_name}"] = float(probs_np[i][c_idx])
            rows.append(row)

    elapsed = time.time() - start_time
    videos_per_sec = total_samples / elapsed if elapsed > 0 else None
    return all_labels, all_preds, pd.DataFrame(rows), elapsed, videos_per_sec

# ============================================================
# 11. RUN ONE EXPERIMENT
# ============================================================
def run_experiment(protocol_name, train_csv_name, val_csv_name, test_csv_name, frame_source, model_name):
    print("\n" + "=" * 100)
    print(f"RUNNING | protocol={protocol_name} | frame_source={frame_source} | model={model_name}")
    print("=" * 100)

    train_df = load_split_df(train_csv_name)
    test_df = load_split_df(test_csv_name)

    if val_csv_name is None:
        train_df = train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
        n_val = max(1, int(0.10 * len(train_df)))
        val_df = train_df.iloc[:n_val].copy().reset_index(drop=True)
        train_df = train_df.iloc[n_val:].copy().reset_index(drop=True)
    else:
        val_df = load_split_df(val_csv_name)

    if LIMIT_TRAIN is not None:
        train_df = train_df.head(LIMIT_TRAIN).copy()
    if LIMIT_VAL is not None:
        val_df = val_df.head(LIMIT_VAL).copy()
    if LIMIT_TEST is not None:
        test_df = test_df.head(LIMIT_TEST).copy()

    train_df = attach_labels(train_df)
    val_df = attach_labels(val_df)
    test_df = attach_labels(test_df)

    train_df = attach_frame_info(train_df, frame_source)
    val_df = attach_frame_info(val_df, frame_source)
    test_df = attach_frame_info(test_df, frame_source)

    print(f"train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

    batch_size = BATCH_SIZE_BY_MODEL[model_name]

    train_loader = DataLoader(
        FrameVotingDataset(train_df, train_transform, NUM_SAMPLE_FRAMES, SAMPLE_START_FRAC, SAMPLE_END_FRAC),
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_fn
    )
    val_loader = DataLoader(
        FrameVotingDataset(val_df, eval_transform, NUM_SAMPLE_FRAMES, SAMPLE_START_FRAC, SAMPLE_END_FRAC),
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_fn
    )
    test_loader = DataLoader(
        FrameVotingDataset(test_df, eval_transform, NUM_SAMPLE_FRAMES, SAMPLE_START_FRAC, SAMPLE_END_FRAC),
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_fn
    )

    run_name = f"{protocol_name}__{frame_source.lower()}__{model_name}__f{NUM_SAMPLE_FRAMES}__img{IMG_SIZE}"
    results_dir = RESULTS_ROOT / run_name
    checkpoint_dir = CHECKPOINT_ROOT / run_name
    log_dir = LOG_ROOT / run_name

    results_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    log_dir.mkdir(parents=True, exist_ok=True)

    metrics_json = results_dir / "metrics.json"
    if SKIP_IF_METRICS_EXISTS and metrics_json.exists():
        print("Skipping, already finished:", run_name)
        with open(metrics_json, "r") as f:
            return json.load(f)

    model = create_model(model_name, NUM_CLASSES, use_pretrained=USE_PRETRAINED).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and device.type == "cuda"))

    best_val_f1 = -1.0
    best_epoch = -1
    epochs_without_improvement = 0
    history = []

    best_ckpt_path = checkpoint_dir / "best_model.pt"
    best_state_dict = None

    train_start = time.time()

    for epoch in range(1, EPOCHS + 1):
        train_metrics = run_one_epoch(model, train_loader, criterion, optimizer, scaler, train=True)
        val_metrics = run_one_epoch(model, val_loader, criterion, optimizer=None, scaler=scaler, train=False)

        history.append({
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
        })

        print(
            f"Epoch {epoch:02d} | "
            f"train_macro_f1={train_metrics['macro_f1']:.4f} | "
            f"val_macro_f1={val_metrics['macro_f1']:.4f}"
        )

        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1 = val_metrics["macro_f1"]
            best_epoch = epoch
            epochs_without_improvement = 0
            best_state_dict = copy.deepcopy(model.state_dict())
            torch.save({
                "epoch": epoch,
                "model_state_dict": best_state_dict,
                "model_name": model_name,
                "frame_source": frame_source,
                "protocol_name": protocol_name,
                "label2id": label2id,
                "id2label": id2label,
            }, best_ckpt_path)
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            print("Early stopping triggered.")
            break

    train_elapsed = time.time() - train_start
    pd.DataFrame(history).to_csv(log_dir / "training_log.csv", index=False)

    if best_state_dict is None:
        raise RuntimeError("No best model was saved.")

    model.load_state_dict(best_state_dict)

    test_labels, test_preds, pred_df, infer_elapsed, videos_per_sec = predict_with_metadata(model, test_loader)

    test_acc = accuracy_score(test_labels, test_preds)
    test_macro_f1 = f1_score(test_labels, test_preds, average="macro")
    test_weighted_f1 = f1_score(test_labels, test_preds, average="weighted")

    cm = confusion_matrix(test_labels, test_preds)
    report_dict = classification_report(
        test_labels,
        test_preds,
        target_names=[id2label[i] for i in range(NUM_CLASSES)],
        output_dict=True,
        zero_division=0
    )
    report_df = pd.DataFrame(report_dict).transpose()

    pred_csv = results_dir / "test_predictions.csv"
    report_csv = results_dir / "classification_report.csv"
    cm_png = results_dir / "confusion_matrix.png"

    pred_df.to_csv(pred_csv, index=False)
    report_df.to_csv(report_csv)

    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation="nearest")
    plt.title(f"{protocol_name} | {frame_source} | {model_name}")
    plt.colorbar()
    ticks = np.arange(NUM_CLASSES)
    plt.xticks(ticks, [id2label[i] for i in range(NUM_CLASSES)], rotation=45, ha="right")
    plt.yticks(ticks, [id2label[i] for i in range(NUM_CLASSES)])
    plt.xlabel("Predicted")
    plt.ylabel("True")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")
    plt.tight_layout()
    plt.savefig(cm_png, dpi=200, bbox_inches="tight")
    plt.close()

    per_class_f1 = {}
    for cls in CLASS_NAMES:
        if cls in report_df.index:
            per_class_f1[cls] = float(report_df.loc[cls, "f1-score"])

    metrics = {
        "protocol_name": protocol_name,
        "frame_source": frame_source,
        "model_name": model_name,
        "num_sample_frames": NUM_SAMPLE_FRAMES,
        "batch_size": batch_size,
        "best_epoch": int(best_epoch),
        "best_val_macro_f1": float(best_val_f1),
        "test_accuracy": float(test_acc),
        "test_macro_f1": float(test_macro_f1),
        "test_weighted_f1": float(test_weighted_f1),
        "per_class_f1": per_class_f1,
        "train_time_minutes": float(train_elapsed / 60.0),
        "inference_time_seconds": float(infer_elapsed),
        "videos_per_second": float(videos_per_sec) if videos_per_sec is not None else None,
        "train_rows": int(len(train_df)),
        "val_rows": int(len(val_df)),
        "test_rows": int(len(test_df)),
        "predictions_csv": str(pred_csv),
        "classification_report_csv": str(report_csv),
        "confusion_matrix_png": str(cm_png),
        "checkpoint_best": str(best_ckpt_path),
    }

    with open(metrics_json, "w") as f:
        json.dump(metrics, f, indent=4)

    print(json.dumps(metrics, indent=2))

    del model, train_loader, val_loader, test_loader, best_state_dict
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metrics

# ============================================================
# 12. RUN ALL EXPERIMENTS
# ============================================================
summary_csv = RESULTS_ROOT / "stage5_rgb_master_summary.csv"

all_metrics = []
if summary_csv.exists():
    try:
        prev = pd.read_csv(summary_csv)
        all_metrics = prev.to_dict(orient="records")
    except Exception:
        all_metrics = []

done_keys = set()
for m in all_metrics:
    done_keys.add((m["protocol_name"], m["frame_source"], m["model_name"]))

for frame_source in FRAME_SOURCES:
    for protocol_name, train_csv_name, val_csv_name, test_csv_name in SPLIT_CONFIGS:
        for model_name in MODELS_TO_RUN:
            key = (protocol_name, frame_source, model_name)
            if SKIP_IF_METRICS_EXISTS and key in done_keys:
                print("Already in summary, skipping:", key)
                continue

            metrics = run_experiment(
                protocol_name=protocol_name,
                train_csv_name=train_csv_name,
                val_csv_name=val_csv_name,
                test_csv_name=test_csv_name,
                frame_source=frame_source,
                model_name=model_name
            )
            all_metrics.append(metrics)
            done_keys.add(key)

            pd.DataFrame(all_metrics).to_csv(summary_csv, index=False)

final_df = pd.DataFrame(all_metrics)
final_df.to_csv(summary_csv, index=False)

print("\nDONE.")
print("Saved summary to:", summary_csv)
display(
    final_df[[
        "protocol_name",
        "frame_source",
        "model_name",
        "best_val_macro_f1",
        "test_accuracy",
        "test_macro_f1",
        "test_weighted_f1",
        "train_time_minutes",
        "videos_per_second"
    ]].sort_values(["protocol_name", "frame_source", "test_macro_f1"], ascending=[True, True, False])
)

ROOT: /data/Sajjan_Singh/gesture_recognition
FRAME_SUMMARY_CSV: /data/Sajjan_Singh/gesture_recognition/data/processed/manifests/frame_extraction_trimmed_summary.csv
CLASS_NAMES: ['doctor', 'emergency', 'fire', 'help', 'robbery', 'sit_down', 'stand_up']
Using device: cuda
GPU name: NVIDIA H100 NVL

RUNNING | protocol=protocol_A | frame_source=RGB | model=resnet18
train=525, val=77, test=147
Skipping, already finished: protocol_A__rgb__resnet18__f32__img224

RUNNING | protocol=protocol_A | frame_source=RGB | model=efficientnet_b0
train=525, val=77, test=147
Skipping, already finished: protocol_A__rgb__efficientnet_b0__f32__img224

RUNNING | protocol=protocol_A | frame_source=RGB | model=mobilenet_v3_small
train=525, val=77, test=147
Epoch 01 | train_macro_f1=0.1201 | val_macro_f1=0.1499
Epoch 02 | train_macro_f1=0.2925 | val_macro_f1=0.1981
Epoch 03 | train_macro_f1=0.3735 | val_macro_f1=0.2236
Epoch 04 | train_macro_f1=0.4832 | val_macro_f1=0.3305
Epoch 05 | train_macro_f1=0.6062 | val_

In [1]:
# ============================================================
# 05_stage5_master_rgb_singlecell.ipynb
# ONE-CELL, RESUMABLE, STABLE STAGE-5 RGB BENCHMARK
# ============================================================

import os
import gc
import json
import time
import copy
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import (
    resnet18, ResNet18_Weights,
    efficientnet_b0, EfficientNet_B0_Weights,
    mobilenet_v3_small, MobileNet_V3_Small_Weights,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)

# ============================================================
# 1. SETTINGS
# ============================================================
SEED = 42

IMG_SIZE = 224
NUM_SAMPLE_FRAMES = 32

# 400 is not practical here. Use stable values.
BATCH_SIZE_BY_MODEL = {
    "resnet18": 24,
    "efficientnet_b0": 16,
    "mobilenet_v3_small": 32,
}

EPOCHS = 12
LR = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 4

USE_AMP = True
USE_PRETRAINED = True

# Jupyter-safe
NUM_WORKERS = 0
PIN_MEMORY = False

# Optional quick debugging
LIMIT_TRAIN = None
LIMIT_VAL = None
LIMIT_TEST = None

# Smart sampling from middle useful region
SAMPLE_START_FRAC = 0.10
SAMPLE_END_FRAC = 0.90

# Resume / skip finished runs automatically
SKIP_IF_METRICS_EXISTS = True

# Run all Stage 5 combinations
FRAME_SOURCES = [
    "RGB",
    "RGB_BGREM",
]

MODELS_TO_RUN = [
    "resnet18",
    "efficientnet_b0",
    "mobilenet_v3_small",
]

# protocol_name, train_csv, val_csv, test_csv
SPLIT_CONFIGS = [
    ("protocol_A",          "protocol_A_train.csv",          "protocol_A_val.csv",          "protocol_A_test.csv"),
    ("protocol_B_4_to_8",   "protocol_B_train_4_test_8.csv", None,                          "protocol_B_test_8.csv"),
    ("protocol_B_8_to_4",   "protocol_B_train_8_test_4.csv", None,                          "protocol_B_test_4.csv"),
    ("protocol_B_4_6_to_8", "protocol_B_train_4_6_test_8.csv", None,                        "protocol_B_test_8_from_4_6.csv"),
    ("protocol_B_4_8_to_6", "protocol_B_train_4_8_test_6.csv", None,                        "protocol_B_test_6_from_4_8.csv"),
    ("protocol_B_6_8_to_4", "protocol_B_train_6_8_test_4.csv", None,                        "protocol_B_test_4_from_6_8.csv"),
]

CLASS_NAMES = [
    "doctor",
    "emergency",
    "fire",
    "help",
    "robbery",
    "sit_down",
    "stand_up",
]
label2id = {c: i for i, c in enumerate(CLASS_NAMES)}
id2label = {i: c for c, i in label2id.items()}
NUM_CLASSES = len(CLASS_NAMES)

# ============================================================
# 2. REPRODUCIBILITY
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ============================================================
# 3. PATHS
# ============================================================
cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd

SPLIT_DIR = ROOT / "data" / "processed" / "splits"
MANIFEST_DIR = ROOT / "data" / "processed" / "manifests"

RESULTS_ROOT = ROOT / "results" / "stage5_rgb_master"
CHECKPOINT_ROOT = ROOT / "checkpoints" / "stage5_rgb_master"
LOG_ROOT = ROOT / "logs" / "stage5_rgb_master"

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
LOG_ROOT.mkdir(parents=True, exist_ok=True)

FRAME_SUMMARY_CSV = MANIFEST_DIR / "frame_extraction_trimmed_summary.csv"
if not FRAME_SUMMARY_CSV.exists():
    raise FileNotFoundError(f"Missing file: {FRAME_SUMMARY_CSV}")

print("ROOT:", ROOT)
print("FRAME_SUMMARY_CSV:", FRAME_SUMMARY_CSV)
print("CLASS_NAMES:", CLASS_NAMES)

# ============================================================
# 4. DEVICE
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
torch.backends.cudnn.benchmark = False

# ============================================================
# 5. FRAME SUMMARY
# ============================================================
frame_df_all = pd.read_csv(FRAME_SUMMARY_CSV)
frame_df_all = frame_df_all[frame_df_all["status"] == "success"].copy()
frame_df_all = frame_df_all[["pair_id", "modality", "output_dir", "frames_extracted"]].drop_duplicates()

# ============================================================
# 6. HELPERS
# ============================================================
def sample_indices_middle_region(n_total, n_samples, start_frac=0.10, end_frac=0.90):
    if n_total <= 0:
        return [0] * n_samples

    start_idx = int(round(start_frac * (n_total - 1)))
    end_idx = int(round(end_frac * (n_total - 1)))

    start_idx = max(0, min(start_idx, n_total - 1))
    end_idx = max(start_idx, min(end_idx, n_total - 1))

    idxs = np.linspace(start_idx, end_idx, n_samples)
    idxs = np.round(idxs).astype(int)
    return np.clip(idxs, 0, n_total - 1).tolist()

def frame_index_to_path(frame_dir, idx):
    return Path(frame_dir) / f"frame_{idx+1:06d}.jpg"

def load_split_df(csv_name):
    csv_path = SPLIT_DIR / csv_name
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing split file: {csv_path}")
    df = pd.read_csv(csv_path)
    if "is_valid_pair" in df.columns:
        df = df[df["is_valid_pair"] == True].copy()
    return df.reset_index(drop=True)

def attach_labels(df_):
    out = df_.copy()
    out = out[out["gesture"].isin(CLASS_NAMES)].copy()
    out["label"] = out["gesture"].map(label2id)
    return out.reset_index(drop=True)

def attach_frame_info(df_, frame_source):
    sub = frame_df_all[frame_df_all["modality"] == frame_source].copy()
    sub = sub.rename(columns={
        "output_dir": "frame_dir",
        "frames_extracted": "num_frames_available"
    })
    sub = sub[["pair_id", "frame_dir", "num_frames_available"]].drop_duplicates()

    out = df_.merge(sub, on="pair_id", how="left")
    out = out[out["num_frames_available"].fillna(0) > 0].copy().reset_index(drop=True)
    out["num_frames_available"] = out["num_frames_available"].astype(int)
    return out

# ============================================================
# 7. TRANSFORMS
# ============================================================
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

# ============================================================
# 8. DATASET
# ============================================================
class FrameVotingDataset(Dataset):
    def __init__(self, df, transform, num_sample_frames=32, start_frac=0.10, end_frac=0.90):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform
        self.sampled_paths = []

        for _, row in self.df.iterrows():
            idxs = sample_indices_middle_region(
                int(row["num_frames_available"]),
                num_sample_frames,
                start_frac,
                end_frac
            )
            self.sampled_paths.append([frame_index_to_path(row["frame_dir"], i) for i in idxs])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        paths = self.sampled_paths[idx]

        frames = []
        for p in paths:
            try:
                with Image.open(p) as f:
                    img = f.convert("RGB")
            except Exception:
                img = Image.fromarray(np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))
            frames.append(self.transform(img))

        frames = torch.stack(frames, dim=0)
        label = int(row["label"])

        meta = {
            "pair_id": row["pair_id"],
            "subject_id": row["subject_id"],
            "distance": row["distance"],
            "gesture": row["gesture"],
            "frame_dir": row["frame_dir"],
        }
        return frames, label, meta

def collate_fn(batch):
    frames = torch.stack([x[0] for x in batch], dim=0)
    labels = torch.tensor([x[1] for x in batch], dtype=torch.long)
    metas = [x[2] for x in batch]
    return frames, labels, metas

# ============================================================
# 9. MODEL FACTORY
# ============================================================
def create_model(model_name, num_classes, use_pretrained=True):
    if model_name == "resnet18":
        model = resnet18(weights=ResNet18_Weights.DEFAULT if use_pretrained else None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif model_name == "efficientnet_b0":
        model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT if use_pretrained else None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    elif model_name == "mobilenet_v3_small":
        model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT if use_pretrained else None)
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)

    else:
        raise ValueError(f"Unsupported model: {model_name}")

    return model

def forward_video_voting(model, frames):
    B, T, C, H, W = frames.shape
    frames = frames.view(B * T, C, H, W)
    frame_logits = model(frames)
    frame_logits = frame_logits.view(B, T, -1)
    video_logits = frame_logits.mean(dim=1)
    return video_logits

# ============================================================
# 10. TRAIN / EVAL
# ============================================================
def run_one_epoch(model, loader, criterion, optimizer=None, scaler=None, train=True):
    model.train() if train else model.eval()

    total_loss = 0.0
    all_labels = []
    all_preds = []

    for frames, labels, metas in loader:
        frames, labels = frames.to(device), labels.to(device)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(USE_AMP and device.type == "cuda")
            ):
                video_logits = forward_video_voting(model, frames)
                loss = criterion(video_logits, labels)

            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        preds = video_logits.argmax(dim=1)

        total_loss += loss.item() * labels.size(0)
        all_labels.extend(labels.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(all_labels, all_preds),
        "macro_f1": f1_score(all_labels, all_preds, average="macro"),
        "weighted_f1": f1_score(all_labels, all_preds, average="weighted"),
    }

@torch.no_grad()
def predict_with_metadata(model, loader):
    model.eval()

    all_labels = []
    all_preds = []
    rows = []

    start_time = time.time()
    total_samples = 0

    for frames, labels, metas in loader:
        frames, labels = frames.to(device), labels.to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(USE_AMP and device.type == "cuda")
        ):
            video_logits = forward_video_voting(model, frames)
            probs = torch.softmax(video_logits, dim=1)
            preds = probs.argmax(dim=1)

        probs_np = probs.detach().cpu().numpy()
        preds_np = preds.detach().cpu().numpy()
        labels_np = labels.detach().cpu().numpy()

        total_samples += len(labels_np)

        all_labels.extend(labels_np.tolist())
        all_preds.extend(preds_np.tolist())

        for i, meta in enumerate(metas):
            row = {
                "pair_id": meta["pair_id"],
                "subject_id": meta["subject_id"],
                "distance": meta["distance"],
                "gesture_true": id2label[int(labels_np[i])],
                "gesture_pred": id2label[int(preds_np[i])],
                "frame_dir": meta["frame_dir"],
            }
            for c_idx, c_name in id2label.items():
                row[f"prob_{c_name}"] = float(probs_np[i][c_idx])
            rows.append(row)

    elapsed = time.time() - start_time
    videos_per_sec = total_samples / elapsed if elapsed > 0 else None
    return all_labels, all_preds, pd.DataFrame(rows), elapsed, videos_per_sec

# ============================================================
# 11. RUN ONE EXPERIMENT
# ============================================================
def run_experiment(protocol_name, train_csv_name, val_csv_name, test_csv_name, frame_source, model_name):
    print("\n" + "=" * 100)
    print(f"RUNNING | protocol={protocol_name} | frame_source={frame_source} | model={model_name}")
    print("=" * 100)

    train_df = load_split_df(train_csv_name)
    test_df = load_split_df(test_csv_name)

    if val_csv_name is None:
        train_df = train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
        n_val = max(1, int(0.10 * len(train_df)))
        val_df = train_df.iloc[:n_val].copy().reset_index(drop=True)
        train_df = train_df.iloc[n_val:].copy().reset_index(drop=True)
    else:
        val_df = load_split_df(val_csv_name)

    if LIMIT_TRAIN is not None:
        train_df = train_df.head(LIMIT_TRAIN).copy()
    if LIMIT_VAL is not None:
        val_df = val_df.head(LIMIT_VAL).copy()
    if LIMIT_TEST is not None:
        test_df = test_df.head(LIMIT_TEST).copy()

    train_df = attach_labels(train_df)
    val_df = attach_labels(val_df)
    test_df = attach_labels(test_df)

    train_df = attach_frame_info(train_df, frame_source)
    val_df = attach_frame_info(val_df, frame_source)
    test_df = attach_frame_info(test_df, frame_source)

    print(f"train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

    batch_size = BATCH_SIZE_BY_MODEL[model_name]

    train_loader = DataLoader(
        FrameVotingDataset(train_df, train_transform, NUM_SAMPLE_FRAMES, SAMPLE_START_FRAC, SAMPLE_END_FRAC),
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_fn
    )
    val_loader = DataLoader(
        FrameVotingDataset(val_df, eval_transform, NUM_SAMPLE_FRAMES, SAMPLE_START_FRAC, SAMPLE_END_FRAC),
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_fn
    )
    test_loader = DataLoader(
        FrameVotingDataset(test_df, eval_transform, NUM_SAMPLE_FRAMES, SAMPLE_START_FRAC, SAMPLE_END_FRAC),
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_fn
    )

    run_name = f"{protocol_name}__{frame_source.lower()}__{model_name}__f{NUM_SAMPLE_FRAMES}__img{IMG_SIZE}"
    results_dir = RESULTS_ROOT / run_name
    checkpoint_dir = CHECKPOINT_ROOT / run_name
    log_dir = LOG_ROOT / run_name

    results_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    log_dir.mkdir(parents=True, exist_ok=True)

    metrics_json = results_dir / "metrics.json"
    if SKIP_IF_METRICS_EXISTS and metrics_json.exists():
        print("Skipping, already finished:", run_name)
        with open(metrics_json, "r") as f:
            return json.load(f)

    model = create_model(model_name, NUM_CLASSES, use_pretrained=USE_PRETRAINED).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and device.type == "cuda"))

    best_val_f1 = -1.0
    best_epoch = -1
    epochs_without_improvement = 0
    history = []

    best_ckpt_path = checkpoint_dir / "best_model.pt"
    best_state_dict = None

    train_start = time.time()

    for epoch in range(1, EPOCHS + 1):
        train_metrics = run_one_epoch(model, train_loader, criterion, optimizer, scaler, train=True)
        val_metrics = run_one_epoch(model, val_loader, criterion, optimizer=None, scaler=scaler, train=False)

        history.append({
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
        })

        print(
            f"Epoch {epoch:02d} | "
            f"train_macro_f1={train_metrics['macro_f1']:.4f} | "
            f"val_macro_f1={val_metrics['macro_f1']:.4f}"
        )

        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1 = val_metrics["macro_f1"]
            best_epoch = epoch
            epochs_without_improvement = 0
            best_state_dict = copy.deepcopy(model.state_dict())
            torch.save({
                "epoch": epoch,
                "model_state_dict": best_state_dict,
                "model_name": model_name,
                "frame_source": frame_source,
                "protocol_name": protocol_name,
                "label2id": label2id,
                "id2label": id2label,
            }, best_ckpt_path)
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            print("Early stopping triggered.")
            break

    train_elapsed = time.time() - train_start
    pd.DataFrame(history).to_csv(log_dir / "training_log.csv", index=False)

    if best_state_dict is None:
        raise RuntimeError("No best model was saved.")

    model.load_state_dict(best_state_dict)

    test_labels, test_preds, pred_df, infer_elapsed, videos_per_sec = predict_with_metadata(model, test_loader)

    test_acc = accuracy_score(test_labels, test_preds)
    test_macro_f1 = f1_score(test_labels, test_preds, average="macro")
    test_weighted_f1 = f1_score(test_labels, test_preds, average="weighted")

    cm = confusion_matrix(test_labels, test_preds)
    report_dict = classification_report(
        test_labels,
        test_preds,
        target_names=[id2label[i] for i in range(NUM_CLASSES)],
        output_dict=True,
        zero_division=0
    )
    report_df = pd.DataFrame(report_dict).transpose()

    pred_csv = results_dir / "test_predictions.csv"
    report_csv = results_dir / "classification_report.csv"
    cm_png = results_dir / "confusion_matrix.png"

    pred_df.to_csv(pred_csv, index=False)
    report_df.to_csv(report_csv)

    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation="nearest")
    plt.title(f"{protocol_name} | {frame_source} | {model_name}")
    plt.colorbar()
    ticks = np.arange(NUM_CLASSES)
    plt.xticks(ticks, [id2label[i] for i in range(NUM_CLASSES)], rotation=45, ha="right")
    plt.yticks(ticks, [id2label[i] for i in range(NUM_CLASSES)])
    plt.xlabel("Predicted")
    plt.ylabel("True")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")
    plt.tight_layout()
    plt.savefig(cm_png, dpi=200, bbox_inches="tight")
    plt.close()

    per_class_f1 = {}
    for cls in CLASS_NAMES:
        if cls in report_df.index:
            per_class_f1[cls] = float(report_df.loc[cls, "f1-score"])

    metrics = {
        "protocol_name": protocol_name,
        "frame_source": frame_source,
        "model_name": model_name,
        "num_sample_frames": NUM_SAMPLE_FRAMES,
        "batch_size": batch_size,
        "best_epoch": int(best_epoch),
        "best_val_macro_f1": float(best_val_f1),
        "test_accuracy": float(test_acc),
        "test_macro_f1": float(test_macro_f1),
        "test_weighted_f1": float(test_weighted_f1),
        "per_class_f1": per_class_f1,
        "train_time_minutes": float(train_elapsed / 60.0),
        "inference_time_seconds": float(infer_elapsed),
        "videos_per_second": float(videos_per_sec) if videos_per_sec is not None else None,
        "train_rows": int(len(train_df)),
        "val_rows": int(len(val_df)),
        "test_rows": int(len(test_df)),
        "predictions_csv": str(pred_csv),
        "classification_report_csv": str(report_csv),
        "confusion_matrix_png": str(cm_png),
        "checkpoint_best": str(best_ckpt_path),
    }

    with open(metrics_json, "w") as f:
        json.dump(metrics, f, indent=4)

    print(json.dumps(metrics, indent=2))

    del model, train_loader, val_loader, test_loader, best_state_dict
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metrics

# ============================================================
# 12. RUN ALL EXPERIMENTS
# ============================================================
summary_csv = RESULTS_ROOT / "stage5_rgb_master_summary.csv"

all_metrics = []
if summary_csv.exists():
    try:
        prev = pd.read_csv(summary_csv)
        all_metrics = prev.to_dict(orient="records")
    except Exception:
        all_metrics = []

done_keys = set()
for m in all_metrics:
    done_keys.add((m["protocol_name"], m["frame_source"], m["model_name"]))

for frame_source in FRAME_SOURCES:
    for protocol_name, train_csv_name, val_csv_name, test_csv_name in SPLIT_CONFIGS:
        for model_name in MODELS_TO_RUN:
            key = (protocol_name, frame_source, model_name)
            if SKIP_IF_METRICS_EXISTS and key in done_keys:
                print("Already in summary, skipping:", key)
                continue

            metrics = run_experiment(
                protocol_name=protocol_name,
                train_csv_name=train_csv_name,
                val_csv_name=val_csv_name,
                test_csv_name=test_csv_name,
                frame_source=frame_source,
                model_name=model_name
            )
            all_metrics.append(metrics)
            done_keys.add(key)

            pd.DataFrame(all_metrics).to_csv(summary_csv, index=False)

final_df = pd.DataFrame(all_metrics)
final_df.to_csv(summary_csv, index=False)

print("\nDONE.")
print("Saved summary to:", summary_csv)
display(
    final_df[[
        "protocol_name",
        "frame_source",
        "model_name",
        "best_val_macro_f1",
        "test_accuracy",
        "test_macro_f1",
        "test_weighted_f1",
        "train_time_minutes",
        "videos_per_second"
    ]].sort_values(["protocol_name", "frame_source", "test_macro_f1"], ascending=[True, True, False])
)

ROOT: /data/Sajjan_Singh/gesture_recognition
FRAME_SUMMARY_CSV: /data/Sajjan_Singh/gesture_recognition/data/processed/manifests/frame_extraction_trimmed_summary.csv
CLASS_NAMES: ['doctor', 'emergency', 'fire', 'help', 'robbery', 'sit_down', 'stand_up']
Using device: cuda
GPU name: NVIDIA H100 NVL
Already in summary, skipping: ('protocol_A', 'RGB', 'resnet18')
Already in summary, skipping: ('protocol_A', 'RGB', 'efficientnet_b0')
Already in summary, skipping: ('protocol_A', 'RGB', 'mobilenet_v3_small')
Already in summary, skipping: ('protocol_B_4_to_8', 'RGB', 'resnet18')
Already in summary, skipping: ('protocol_B_4_to_8', 'RGB', 'efficientnet_b0')
Already in summary, skipping: ('protocol_B_4_to_8', 'RGB', 'mobilenet_v3_small')
Already in summary, skipping: ('protocol_B_8_to_4', 'RGB', 'resnet18')
Already in summary, skipping: ('protocol_B_8_to_4', 'RGB', 'efficientnet_b0')
Already in summary, skipping: ('protocol_B_8_to_4', 'RGB', 'mobilenet_v3_small')
Already in summary, skipping: ('p

,protocol_name,frame_source,model_name,best_val_macro_f1,test_accuracy,test_macro_f1,test_weighted_f1,train_time_minutes,videos_per_second
1,protocol_A,RGB,efficientnet_b0,0.841927,0.884354,0.885118,0.885118,135.243664,0.972483
0,protocol_A,RGB,resnet18,0.894154,0.809524,0.807739,0.807739,148.005852,0.456316
2,protocol_A,RGB,mobilenet_v3_small,0.570384,0.619048,0.622501,0.622501,150.136283,0.476967
18,protocol_A,RGB_BGREM,resnet18,0.842454,0.829932,0.828292,0.828292,87.946307,0.471911
19,protocol_A,RGB_BGREM,efficientnet_b0,0.858412,0.816327,0.819876,0.819876,120.517467,0.415251
20,protocol_A,RGB_BGREM,mobilenet_v3_small,0.490909,0.510204,0.481947,0.481947,193.397623,0.402600
10,protocol_B_4_6_to_8,RGB,efficientnet_b0,0.815767,0.560150,0.559856,0.559856,67.651529,0.680196
9,protocol_B_4_6_to_8,RGB,resnet18,0.737710,0.552632,0.552900,0.552900,76.987336,0.614342
11,protocol_B_4_6_to_8,RGB,mobilenet_v3_small,0.610794,0.327068,0.294327,0.294327,80.402574,0.671377
28,protocol_B_4_6_to_8,RGB_BGREM,efficientnet_b0,0.756854,0.624060,0.624181,0.624181,98.401112,0.410089


In [1]:
# ============================================================
# CREATE ALL REMAINING PROTOCOL CSVs
# ============================================================
# Input:
#   data/processed/manifests/paired_manifest_with_subjects.csv
#
# Output:
#   data/processed/splits/*.csv
#   data/processed/splits/all_protocols_summary.csv
# ============================================================

import json
from pathlib import Path
import pandas as pd

# ============================================================
# 1. PATHS
# ============================================================
cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd

MANIFEST_PATH = ROOT / "data" / "processed" / "manifests" / "paired_manifest_with_subjects.csv"
SPLIT_DIR = ROOT / "data" / "processed" / "splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Missing manifest: {MANIFEST_PATH}")

print("ROOT:", ROOT)
print("MANIFEST_PATH:", MANIFEST_PATH)
print("SPLIT_DIR:", SPLIT_DIR)

# ============================================================
# 2. LOAD MANIFEST
# ============================================================
df = pd.read_csv(MANIFEST_PATH)

if "is_valid_pair" in df.columns:
    df = df[df["is_valid_pair"] == True].copy()

required_cols = ["pair_id", "subject_id", "gesture", "distance"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

df = df.reset_index(drop=True)

print("\nLoaded manifest shape:", df.shape)
print("Unique distances:", sorted(df["distance"].dropna().unique().tolist()))
print("Unique gestures:", sorted(df["gesture"].dropna().unique().tolist()))
print("Unique subjects:", df["subject_id"].nunique())

print("\nCount by distance:")
print(df["distance"].value_counts().sort_index())

# ============================================================
# 3. DISTANCE NORMALIZATION
# ============================================================
DISTANCE_ALIASES = {
    "4_feet": "4_feet",
    "4 feet": "4_feet",
    "6_feet": "6_feet",
    "6 feet": "6_feet",
    "8_feet": "8_feet",
    "8 feet": "8_feet",
}

df["distance"] = df["distance"].map(lambda x: DISTANCE_ALIASES.get(str(x), str(x)))

VALID_DISTANCES = ["4_feet", "6_feet", "8_feet"]
df = df[df["distance"].isin(VALID_DISTANCES)].copy().reset_index(drop=True)

print("\nAfter normalization:")
print(df["distance"].value_counts().sort_index())

# ============================================================
# 4. HELPER TO SAVE SPLITS
# ============================================================
def save_csv_if_missing(df_out, filename):
    path = SPLIT_DIR / filename
    if path.exists():
        return path, False
    df_out.to_csv(path, index=False)
    return path, True

def make_name(dist_list):
    # ["4_feet"] -> "4"
    # ["4_feet","6_feet"] -> "4_6"
    # ["4_feet","6_feet","8_feet"] -> "4_6_8"
    return "_".join([d.split("_")[0] for d in dist_list])

# ============================================================
# 5. DEFINE FULL PROTOCOL SET
# ============================================================
# We will create:
# 1-distance train -> all 3 tests
# 2-distance train -> all 3 tests
# 3-distance train -> all 3 tests

full_protocols = [
    # single-distance train
    (["4_feet"], ["4_feet"]),
    (["4_feet"], ["6_feet"]),
    (["4_feet"], ["8_feet"]),

    (["6_feet"], ["4_feet"]),
    (["6_feet"], ["6_feet"]),
    (["6_feet"], ["8_feet"]),

    (["8_feet"], ["4_feet"]),
    (["8_feet"], ["6_feet"]),
    (["8_feet"], ["8_feet"]),

    # two-distance train
    (["4_feet", "6_feet"], ["4_feet"]),
    (["4_feet", "6_feet"], ["6_feet"]),
    (["4_feet", "6_feet"], ["8_feet"]),

    (["4_feet", "8_feet"], ["4_feet"]),
    (["4_feet", "8_feet"], ["6_feet"]),
    (["4_feet", "8_feet"], ["8_feet"]),

    (["6_feet", "8_feet"], ["4_feet"]),
    (["6_feet", "8_feet"], ["6_feet"]),
    (["6_feet", "8_feet"], ["8_feet"]),

    # three-distance train
    (["4_feet", "6_feet", "8_feet"], ["4_feet"]),
    (["4_feet", "6_feet", "8_feet"], ["6_feet"]),
    (["4_feet", "6_feet", "8_feet"], ["8_feet"]),
]

# ============================================================
# 6. CREATE SPLITS
# ============================================================
rows = []

for train_distances, test_distances in full_protocols:
    train_name = make_name(train_distances)
    test_name = make_name(test_distances)

    train_df = df[df["distance"].isin(train_distances)].copy().reset_index(drop=True)
    test_df = df[df["distance"].isin(test_distances)].copy().reset_index(drop=True)

    train_file = f"protocol_B_train_{train_name}_test_{test_name}.csv"

    # test filenames keep compatibility with your earlier naming style
    if len(train_distances) == 1 and train_distances == test_distances:
        test_file = f"protocol_B_test_{test_name}_same.csv"
    elif len(train_distances) == 1:
        test_file = f"protocol_B_test_{test_name}.csv"
    else:
        test_file = f"protocol_B_test_{test_name}_from_{train_name}.csv"

    train_path, train_created = save_csv_if_missing(train_df, train_file)
    test_path, test_created = save_csv_if_missing(test_df, test_file)

    rows.append({
        "train_distances": ",".join(train_distances),
        "test_distances": ",".join(test_distances),
        "train_file": str(train_path),
        "test_file": str(test_path),
        "train_rows": len(train_df),
        "test_rows": len(test_df),
        "train_subjects": train_df["subject_id"].nunique(),
        "test_subjects": test_df["subject_id"].nunique(),
        "train_created_now": train_created,
        "test_created_now": test_created,
    })

summary_df = pd.DataFrame(rows)

# ============================================================
# 7. SAVE SUMMARY
# ============================================================
summary_csv = SPLIT_DIR / "all_protocols_summary.csv"
summary_df.to_csv(summary_csv, index=False)

print("\nSaved protocol summary to:", summary_csv)

print("\nAll protocol rows:")
display(summary_df)

print("\nNewly created files only:")
display(summary_df[(summary_df["train_created_now"] == True) | (summary_df["test_created_now"] == True)])

print("\nDone.")

ROOT: /data/Sajjan_Singh/gesture_recognition
MANIFEST_PATH: /data/Sajjan_Singh/gesture_recognition/data/processed/manifests/paired_manifest_with_subjects.csv
SPLIT_DIR: /data/Sajjan_Singh/gesture_recognition/data/processed/splits

Loaded manifest shape: (749, 26)
Unique distances: ['4_feet', '6_feet', '8_feet']
Unique gestures: ['doctor', 'emergency', 'fire', 'help', 'robbery', 'sit_down', 'stand_up']
Unique subjects: 107

Count by distance:
distance
4_feet    238
6_feet    245
8_feet    266
Name: count, dtype: int64

After normalization:
distance
4_feet    238
6_feet    245
8_feet    266
Name: count, dtype: int64

Saved protocol summary to: /data/Sajjan_Singh/gesture_recognition/data/processed/splits/all_protocols_summary.csv

All protocol rows:


,train_distances,test_distances,train_file,test_file,train_rows,test_rows,train_subjects,test_subjects,train_created_now,test_created_now
0,4_feet,4_feet,/data/Sajjan_Singh/gesture_recognition/data/pr...,/data/Sajjan_Singh/gesture_recognition/data/pr...,238,238,34,34,False,False
1,4_feet,6_feet,/data/Sajjan_Singh/gesture_recognition/data/pr...,/data/Sajjan_Singh/gesture_recognition/data/pr...,238,245,34,35,True,True
2,4_feet,8_feet,/data/Sajjan_Singh/gesture_recognition/data/pr...,/data/Sajjan_Singh/gesture_recognition/data/pr...,238,266,34,38,False,False
3,6_feet,4_feet,/data/Sajjan_Singh/gesture_recognition/data/pr...,/data/Sajjan_Singh/gesture_recognition/data/pr...,245,238,35,34,True,False
4,6_feet,6_feet,/data/Sajjan_Singh/gesture_recognition/data/pr...,/data/Sajjan_Singh/gesture_recognition/data/pr...,245,245,35,35,False,False
5,6_feet,8_feet,/data/Sajjan_Singh/gesture_recognition/data/pr...,/data/Sajjan_Singh/gesture_recognition/data/pr...,245,266,35,38,True,False
6,8_feet,4_feet,/data/Sajjan_Singh/gesture_recognition/data/pr...,/data/Sajjan_Singh/gesture_recognition/data/pr...,266,238,38,34,False,False
7,8_feet,6_feet,/data/Sajjan_Singh/gesture_recognition/data/pr...,/data/Sajjan_Singh/gesture_recognition/data/pr...,266,245,38,35,True,False
8,8_feet,8_feet,/data/Sajjan_Singh/gesture_recognition/data/pr...,/data/Sajjan_Singh/gesture_recognition/data/pr...,266,266,38,38,False,False
9,"4_feet,6_feet",4_feet,/data/Sajjan_Singh/gesture_recognition/data/pr...,/data/Sajjan_Singh/gesture_recognition/data/pr...,483,238,69,34,True,True



Newly created files only:


,train_distances,test_distances,train_file,test_file,train_rows,test_rows,train_subjects,test_subjects,train_created_now,test_created_now
1,4_feet,6_feet,/data/Sajjan_Singh/gesture_recognition/data/pr...,/data/Sajjan_Singh/gesture_recognition/data/pr...,238,245,34,35,True,True
3,6_feet,4_feet,/data/Sajjan_Singh/gesture_recognition/data/pr...,/data/Sajjan_Singh/gesture_recognition/data/pr...,245,238,35,34,True,False
5,6_feet,8_feet,/data/Sajjan_Singh/gesture_recognition/data/pr...,/data/Sajjan_Singh/gesture_recognition/data/pr...,245,266,35,38,True,False
7,8_feet,6_feet,/data/Sajjan_Singh/gesture_recognition/data/pr...,/data/Sajjan_Singh/gesture_recognition/data/pr...,266,245,38,35,True,False
9,"4_feet,6_feet",4_feet,/data/Sajjan_Singh/gesture_recognition/data/pr...,/data/Sajjan_Singh/gesture_recognition/data/pr...,483,238,69,34,True,True
10,"4_feet,6_feet",6_feet,/data/Sajjan_Singh/gesture_recognition/data/pr...,/data/Sajjan_Singh/gesture_recognition/data/pr...,483,245,69,35,True,True
12,"4_feet,8_feet",4_feet,/data/Sajjan_Singh/gesture_recognition/data/pr...,/data/Sajjan_Singh/gesture_recognition/data/pr...,504,238,72,34,True,True
14,"4_feet,8_feet",8_feet,/data/Sajjan_Singh/gesture_recognition/data/pr...,/data/Sajjan_Singh/gesture_recognition/data/pr...,504,266,72,38,True,True
16,"6_feet,8_feet",6_feet,/data/Sajjan_Singh/gesture_recognition/data/pr...,/data/Sajjan_Singh/gesture_recognition/data/pr...,511,245,73,35,True,True
17,"6_feet,8_feet",8_feet,/data/Sajjan_Singh/gesture_recognition/data/pr...,/data/Sajjan_Singh/gesture_recognition/data/pr...,511,266,73,38,True,True



Done.


In [ ]:
# ============================================================
# 05_stage5_remaining_protocols_only.ipynb
# TRAIN ONLY THE REMAINING RGB PROTOCOLS
# ============================================================

import os
import gc
import json
import time
import copy
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import (
    resnet18, ResNet18_Weights,
    efficientnet_b0, EfficientNet_B0_Weights,
    mobilenet_v3_small, MobileNet_V3_Small_Weights,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)

# ============================================================
# 1. SETTINGS
# ============================================================
SEED = 42

IMG_SIZE = 224
NUM_SAMPLE_FRAMES = 32

BATCH_SIZE_BY_MODEL = {
    "resnet18": 24,
    "efficientnet_b0": 16,
    "mobilenet_v3_small": 32,
}

EPOCHS = 12
LR = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 4

USE_AMP = True
USE_PRETRAINED = True

NUM_WORKERS = 0
PIN_MEMORY = False

LIMIT_TRAIN = None
LIMIT_VAL = None
LIMIT_TEST = None

SAMPLE_START_FRAC = 0.10
SAMPLE_END_FRAC = 0.90

SKIP_IF_METRICS_EXISTS = True

FRAME_SOURCES = [
    "RGB",
    "RGB_BGREM",
]

MODELS_TO_RUN = [
    "resnet18",
    "efficientnet_b0",
    "mobilenet_v3_small",
]

# ============================================================
# ONLY REMAINING PROTOCOLS
# ============================================================
REMAINING_SPLIT_CONFIGS = [
    ("protocol_B_4_to_4",     "protocol_B_train_4_test_4.csv",     None, "protocol_B_test_4_same.csv"),
    ("protocol_B_4_to_6",     "protocol_B_train_4_test_6.csv",     None, "protocol_B_test_6.csv"),
    ("protocol_B_6_to_6",     "protocol_B_train_6_test_6.csv",     None, "protocol_B_test_6_same.csv"),
    ("protocol_B_6_to_8",     "protocol_B_train_6_test_8.csv",     None, "protocol_B_test_8.csv"),
    ("protocol_B_8_to_6",     "protocol_B_train_8_test_6.csv",     None, "protocol_B_test_6.csv"),
    ("protocol_B_8_to_8",     "protocol_B_train_8_test_8.csv",     None, "protocol_B_test_8_same.csv"),

    ("protocol_B_4_6_to_4",   "protocol_B_train_4_6_test_4.csv",   None, "protocol_B_test_4_from_4_6.csv"),
    ("protocol_B_4_6_to_6",   "protocol_B_train_4_6_test_6.csv",   None, "protocol_B_test_6_from_4_6.csv"),

    ("protocol_B_4_8_to_4",   "protocol_B_train_4_8_test_4.csv",   None, "protocol_B_test_4_from_4_8.csv"),
    ("protocol_B_4_8_to_8",   "protocol_B_train_4_8_test_8.csv",   None, "protocol_B_test_8_from_4_8.csv"),

    ("protocol_B_6_8_to_6",   "protocol_B_train_6_8_test_6.csv",   None, "protocol_B_test_6_from_6_8.csv"),
    ("protocol_B_6_8_to_8",   "protocol_B_train_6_8_test_8.csv",   None, "protocol_B_test_8_from_6_8.csv"),

    ("protocol_B_4_6_8_to_4", "protocol_B_train_4_6_8_test_4.csv", None, "protocol_B_test_4_from_4_6_8.csv"),
    ("protocol_B_4_6_8_to_6", "protocol_B_train_4_6_8_test_6.csv", None, "protocol_B_test_6_from_4_6_8.csv"),
    ("protocol_B_4_6_8_to_8", "protocol_B_train_4_6_8_test_8.csv", None, "protocol_B_test_8_from_4_6_8.csv"),
]

CLASS_NAMES = [
    "doctor",
    "emergency",
    "fire",
    "help",
    "robbery",
    "sit_down",
    "stand_up",
]
label2id = {c: i for i, c in enumerate(CLASS_NAMES)}
id2label = {i: c for c, i in label2id.items()}
NUM_CLASSES = len(CLASS_NAMES)

# ============================================================
# 2. REPRODUCIBILITY
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ============================================================
# 3. PATHS
# ============================================================
cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd

SPLIT_DIR = ROOT / "data" / "processed" / "splits"
MANIFEST_DIR = ROOT / "data" / "processed" / "manifests"

RESULTS_ROOT = ROOT / "results" / "stage5_remaining_protocols"
CHECKPOINT_ROOT = ROOT / "checkpoints" / "stage5_remaining_protocols"
LOG_ROOT = ROOT / "logs" / "stage5_remaining_protocols"

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
LOG_ROOT.mkdir(parents=True, exist_ok=True)

FRAME_SUMMARY_CSV = MANIFEST_DIR / "frame_extraction_trimmed_summary.csv"
if not FRAME_SUMMARY_CSV.exists():
    raise FileNotFoundError(f"Missing file: {FRAME_SUMMARY_CSV}")

print("ROOT:", ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)
print("FRAME_SUMMARY_CSV:", FRAME_SUMMARY_CSV)

# ============================================================
# 4. DEVICE
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
torch.backends.cudnn.benchmark = False

# ============================================================
# 5. FRAME SUMMARY
# ============================================================
frame_df_all = pd.read_csv(FRAME_SUMMARY_CSV)
frame_df_all = frame_df_all[frame_df_all["status"] == "success"].copy()
frame_df_all = frame_df_all[["pair_id", "modality", "output_dir", "frames_extracted"]].drop_duplicates()

# ============================================================
# 6. HELPERS
# ============================================================
def sample_indices_middle_region(n_total, n_samples, start_frac=0.10, end_frac=0.90):
    if n_total <= 0:
        return [0] * n_samples
    start_idx = int(round(start_frac * (n_total - 1)))
    end_idx = int(round(end_frac * (n_total - 1)))
    start_idx = max(0, min(start_idx, n_total - 1))
    end_idx = max(start_idx, min(end_idx, n_total - 1))
    idxs = np.linspace(start_idx, end_idx, n_samples)
    idxs = np.round(idxs).astype(int)
    return np.clip(idxs, 0, n_total - 1).tolist()

def frame_index_to_path(frame_dir, idx):
    return Path(frame_dir) / f"frame_{idx+1:06d}.jpg"

def load_split_df(csv_name):
    csv_path = SPLIT_DIR / csv_name
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing split file: {csv_path}")
    df = pd.read_csv(csv_path)
    if "is_valid_pair" in df.columns:
        df = df[df["is_valid_pair"] == True].copy()
    return df.reset_index(drop=True)

def attach_labels(df_):
    out = df_.copy()
    out = out[out["gesture"].isin(CLASS_NAMES)].copy()
    out["label"] = out["gesture"].map(label2id)
    return out.reset_index(drop=True)

def attach_frame_info(df_, frame_source):
    sub = frame_df_all[frame_df_all["modality"] == frame_source].copy()
    sub = sub.rename(columns={
        "output_dir": "frame_dir",
        "frames_extracted": "num_frames_available"
    })
    sub = sub[["pair_id", "frame_dir", "num_frames_available"]].drop_duplicates()

    out = df_.merge(sub, on="pair_id", how="left")
    out = out[out["num_frames_available"].fillna(0) > 0].copy().reset_index(drop=True)
    out["num_frames_available"] = out["num_frames_available"].astype(int)
    return out

# ============================================================
# 7. TRANSFORMS
# ============================================================
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ============================================================
# 8. DATASET
# ============================================================
class FrameVotingDataset(Dataset):
    def __init__(self, df, transform, num_sample_frames=32, start_frac=0.10, end_frac=0.90):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform
        self.sampled_paths = []

        print(f"Building sampled frame paths for {len(self.df)} samples...")
        for _, row in self.df.iterrows():
            idxs = sample_indices_middle_region(
                int(row["num_frames_available"]),
                num_sample_frames,
                start_frac,
                end_frac
            )
            self.sampled_paths.append([frame_index_to_path(row["frame_dir"], i) for i in idxs])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        paths = self.sampled_paths[idx]

        frames = []
        for p in paths:
            try:
                with Image.open(p) as f:
                    img = f.convert("RGB")
            except Exception:
                img = Image.fromarray(np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))
            frames.append(self.transform(img))

        frames = torch.stack(frames, dim=0)
        label = int(row["label"])

        meta = {
            "pair_id": row["pair_id"],
            "subject_id": row["subject_id"],
            "distance": row["distance"],
            "gesture": row["gesture"],
            "frame_dir": row["frame_dir"],
        }
        return frames, label, meta

def collate_fn(batch):
    frames = torch.stack([x[0] for x in batch], dim=0)
    labels = torch.tensor([x[1] for x in batch], dtype=torch.long)
    metas = [x[2] for x in batch]
    return frames, labels, metas

# ============================================================
# 9. MODEL FACTORY
# ============================================================
def create_model(model_name, num_classes, use_pretrained=True):
    if model_name == "resnet18":
        model = resnet18(weights=ResNet18_Weights.DEFAULT if use_pretrained else None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == "efficientnet_b0":
        model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT if use_pretrained else None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif model_name == "mobilenet_v3_small":
        model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT if use_pretrained else None)
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)
    else:
        raise ValueError(f"Unsupported model: {model_name}")
    return model

def forward_video_voting(model, frames):
    B, T, C, H, W = frames.shape
    frames = frames.view(B * T, C, H, W)
    frame_logits = model(frames)
    frame_logits = frame_logits.view(B, T, -1)
    return frame_logits.mean(dim=1)

# ============================================================
# 10. TRAIN / EVAL
# ============================================================
def run_one_epoch(model, loader, criterion, optimizer=None, scaler=None, train=True):
    model.train() if train else model.eval()

    total_loss = 0.0
    all_labels = []
    all_preds = []

    total_batches = len(loader)

    for batch_idx, (frames, labels, metas) in enumerate(loader):
        frames, labels = frames.to(device), labels.to(device)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(USE_AMP and device.type == "cuda")):
                logits = forward_video_voting(model, frames)
                loss = criterion(logits, labels)

            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        preds = logits.argmax(dim=1)

        total_loss += loss.item() * labels.size(0)
        all_labels.extend(labels.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())

        if batch_idx % 10 == 0 or batch_idx == total_batches - 1:
            mode = "train" if train else "val"
            print(f"   [{mode}] batch {batch_idx+1}/{total_batches} | loss={loss.item():.4f}")

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(all_labels, all_preds),
        "macro_f1": f1_score(all_labels, all_preds, average="macro"),
        "weighted_f1": f1_score(all_labels, all_preds, average="weighted"),
    }

@torch.no_grad()
def predict_with_metadata(model, loader):
    model.eval()

    all_labels = []
    all_preds = []
    rows = []

    start_time = time.time()
    total_samples = 0

    for frames, labels, metas in loader:
        frames, labels = frames.to(device), labels.to(device)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(USE_AMP and device.type == "cuda")):
            logits = forward_video_voting(model, frames)
            probs = torch.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)

        probs_np = probs.detach().cpu().numpy()
        preds_np = preds.detach().cpu().numpy()
        labels_np = labels.detach().cpu().numpy()

        total_samples += len(labels_np)
        all_labels.extend(labels_np.tolist())
        all_preds.extend(preds_np.tolist())

        for i, meta in enumerate(metas):
            row = {
                "pair_id": meta["pair_id"],
                "subject_id": meta["subject_id"],
                "distance": meta["distance"],
                "gesture_true": id2label[int(labels_np[i])],
                "gesture_pred": id2label[int(preds_np[i])],
                "frame_dir": meta["frame_dir"],
            }
            for c_idx, c_name in id2label.items():
                row[f"prob_{c_name}"] = float(probs_np[i][c_idx])
            rows.append(row)

    elapsed = time.time() - start_time
    videos_per_sec = total_samples / elapsed if elapsed > 0 else None
    return all_labels, all_preds, pd.DataFrame(rows), elapsed, videos_per_sec

# ============================================================
# 11. RUN ONE EXPERIMENT
# ============================================================
def run_experiment(protocol_name, train_csv_name, val_csv_name, test_csv_name, frame_source, model_name):
    print("\n" + "=" * 100)
    print(f"RUNNING | protocol={protocol_name} | frame_source={frame_source} | model={model_name}")
    print("=" * 100)

    train_df = load_split_df(train_csv_name)
    test_df = load_split_df(test_csv_name)

    if val_csv_name is None:
        train_df = train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
        n_val = max(1, int(0.10 * len(train_df)))
        val_df = train_df.iloc[:n_val].copy().reset_index(drop=True)
        train_df = train_df.iloc[n_val:].copy().reset_index(drop=True)
    else:
        val_df = load_split_df(val_csv_name)

    train_df = attach_labels(train_df)
    val_df = attach_labels(val_df)
    test_df = attach_labels(test_df)

    train_df = attach_frame_info(train_df, frame_source)
    val_df = attach_frame_info(val_df, frame_source)
    test_df = attach_frame_info(test_df, frame_source)

    if LIMIT_TRAIN is not None:
        train_df = train_df.head(LIMIT_TRAIN).copy()
    if LIMIT_VAL is not None:
        val_df = val_df.head(LIMIT_VAL).copy()
    if LIMIT_TEST is not None:
        test_df = test_df.head(LIMIT_TEST).copy()

    print(f"train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

    batch_size = BATCH_SIZE_BY_MODEL[model_name]

    train_loader = DataLoader(
        FrameVotingDataset(train_df, train_transform, NUM_SAMPLE_FRAMES, SAMPLE_START_FRAC, SAMPLE_END_FRAC),
        batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, collate_fn=collate_fn
    )
    val_loader = DataLoader(
        FrameVotingDataset(val_df, eval_transform, NUM_SAMPLE_FRAMES, SAMPLE_START_FRAC, SAMPLE_END_FRAC),
        batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, collate_fn=collate_fn
    )
    test_loader = DataLoader(
        FrameVotingDataset(test_df, eval_transform, NUM_SAMPLE_FRAMES, SAMPLE_START_FRAC, SAMPLE_END_FRAC),
        batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, collate_fn=collate_fn
    )

    run_name = f"{protocol_name}__{frame_source.lower()}__{model_name}__f{NUM_SAMPLE_FRAMES}__img{IMG_SIZE}"
    results_dir = RESULTS_ROOT / run_name
    checkpoint_dir = CHECKPOINT_ROOT / run_name
    log_dir = LOG_ROOT / run_name

    results_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    log_dir.mkdir(parents=True, exist_ok=True)

    metrics_json = results_dir / "metrics.json"
    if SKIP_IF_METRICS_EXISTS and metrics_json.exists():
        print("Skipping, already finished:", run_name)
        with open(metrics_json, "r") as f:
            return json.load(f)

    model = create_model(model_name, NUM_CLASSES, use_pretrained=USE_PRETRAINED).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and device.type == "cuda"))

    best_val_f1 = -1.0
    best_epoch = -1
    epochs_without_improvement = 0
    history = []
    best_state_dict = None
    best_ckpt_path = checkpoint_dir / "best_model.pt"

    train_start = time.time()

    for epoch in range(1, EPOCHS + 1):
        print(f"\nEpoch {epoch}/{EPOCHS}")
        train_metrics = run_one_epoch(model, train_loader, criterion, optimizer, scaler, train=True)
        val_metrics = run_one_epoch(model, val_loader, criterion, optimizer=None, scaler=scaler, train=False)

        history.append({
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
        })

        print(
            f"Epoch {epoch:02d} | "
            f"train_macro_f1={train_metrics['macro_f1']:.4f} | "
            f"val_macro_f1={val_metrics['macro_f1']:.4f}"
        )

        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1 = val_metrics["macro_f1"]
            best_epoch = epoch
            epochs_without_improvement = 0
            best_state_dict = copy.deepcopy(model.state_dict())
            torch.save({
                "epoch": epoch,
                "model_state_dict": best_state_dict,
                "model_name": model_name,
                "frame_source": frame_source,
                "protocol_name": protocol_name,
                "label2id": label2id,
                "id2label": id2label,
            }, best_ckpt_path)
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            print("Early stopping triggered.")
            break

    train_elapsed = time.time() - train_start
    pd.DataFrame(history).to_csv(log_dir / "training_log.csv", index=False)

    if best_state_dict is None:
        raise RuntimeError("No best model was saved.")

    model.load_state_dict(best_state_dict)

    test_labels, test_preds, pred_df, infer_elapsed, videos_per_sec = predict_with_metadata(model, test_loader)

    test_acc = accuracy_score(test_labels, test_preds)
    test_macro_f1 = f1_score(test_labels, test_preds, average="macro")
    test_weighted_f1 = f1_score(test_labels, test_preds, average="weighted")

    cm = confusion_matrix(test_labels, test_preds)
    report_dict = classification_report(
        test_labels, test_preds,
        target_names=[id2label[i] for i in range(NUM_CLASSES)],
        output_dict=True,
        zero_division=0
    )
    report_df = pd.DataFrame(report_dict).transpose()

    pred_csv = results_dir / "test_predictions.csv"
    report_csv = results_dir / "classification_report.csv"
    cm_png = results_dir / "confusion_matrix.png"

    pred_df.to_csv(pred_csv, index=False)
    report_df.to_csv(report_csv)

    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation="nearest")
    plt.title(f"{protocol_name} | {frame_source} | {model_name}")
    plt.colorbar()
    ticks = np.arange(NUM_CLASSES)
    plt.xticks(ticks, [id2label[i] for i in range(NUM_CLASSES)], rotation=45, ha="right")
    plt.yticks(ticks, [id2label[i] for i in range(NUM_CLASSES)])
    plt.xlabel("Predicted")
    plt.ylabel("True")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")
    plt.tight_layout()
    plt.savefig(cm_png, dpi=200, bbox_inches="tight")
    plt.close()

    per_class_f1 = {}
    for cls in CLASS_NAMES:
        if cls in report_df.index:
            per_class_f1[cls] = float(report_df.loc[cls, "f1-score"])

    metrics = {
        "protocol_name": protocol_name,
        "frame_source": frame_source,
        "model_name": model_name,
        "num_sample_frames": NUM_SAMPLE_FRAMES,
        "batch_size": batch_size,
        "best_epoch": int(best_epoch),
        "best_val_macro_f1": float(best_val_f1),
        "test_accuracy": float(test_acc),
        "test_macro_f1": float(test_macro_f1),
        "test_weighted_f1": float(test_weighted_f1),
        "per_class_f1": per_class_f1,
        "train_time_minutes": float(train_elapsed / 60.0),
        "inference_time_seconds": float(infer_elapsed),
        "videos_per_second": float(videos_per_sec) if videos_per_sec is not None else None,
        "train_rows": int(len(train_df)),
        "val_rows": int(len(val_df)),
        "test_rows": int(len(test_df)),
        "predictions_csv": str(pred_csv),
        "classification_report_csv": str(report_csv),
        "confusion_matrix_png": str(cm_png),
        "checkpoint_best": str(best_ckpt_path),
    }

    with open(metrics_json, "w") as f:
        json.dump(metrics, f, indent=4)

    print(json.dumps(metrics, indent=2))

    del model, train_loader, val_loader, test_loader, best_state_dict
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metrics

# ============================================================
# 12. RUN ONLY REMAINING PROTOCOLS
# ============================================================
summary_csv = RESULTS_ROOT / "remaining_protocols_summary.csv"

all_metrics = []
if summary_csv.exists():
    try:
        prev = pd.read_csv(summary_csv)
        all_metrics = prev.to_dict(orient="records")
    except Exception:
        all_metrics = []

done_keys = set((m["protocol_name"], m["frame_source"], m["model_name"]) for m in all_metrics)

for frame_source in FRAME_SOURCES:
    for protocol_name, train_csv_name, val_csv_name, test_csv_name in REMAINING_SPLIT_CONFIGS:
        for model_name in MODELS_TO_RUN:
            key = (protocol_name, frame_source, model_name)
            if SKIP_IF_METRICS_EXISTS and key in done_keys:
                print("Already in summary, skipping:", key)
                continue

            metrics = run_experiment(
                protocol_name=protocol_name,
                train_csv_name=train_csv_name,
                val_csv_name=val_csv_name,
                test_csv_name=test_csv_name,
                frame_source=frame_source,
                model_name=model_name
            )
            all_metrics.append(metrics)
            done_keys.add(key)
            pd.DataFrame(all_metrics).to_csv(summary_csv, index=False)

final_df = pd.DataFrame(all_metrics)
final_df.to_csv(summary_csv, index=False)

print("\nDONE.")
print("Saved summary to:", summary_csv)
display(final_df[[
    "protocol_name",
    "frame_source",
    "model_name",
    "best_val_macro_f1",
    "test_accuracy",
    "test_macro_f1",
    "test_weighted_f1",
    "train_time_minutes",
    "videos_per_second"
]].sort_values(["protocol_name", "frame_source", "test_macro_f1"], ascending=[True, True, False]))

ROOT: /data/Sajjan_Singh/gesture_recognition
RESULTS_ROOT: /data/Sajjan_Singh/gesture_recognition/results/stage5_remaining_protocols
FRAME_SUMMARY_CSV: /data/Sajjan_Singh/gesture_recognition/data/processed/manifests/frame_extraction_trimmed_summary.csv
Using device: cuda
GPU name: NVIDIA H100 NVL

RUNNING | protocol=protocol_B_4_to_4 | frame_source=RGB | model=resnet18
train=215, val=23, test=238
Building sampled frame paths for 215 samples...
Building sampled frame paths for 23 samples...
Building sampled frame paths for 238 samples...

Epoch 1/12
   [train] batch 1/9 | loss=2.1320
   [train] batch 9/9 | loss=1.6647
   [val] batch 1/1 | loss=1.7063
Epoch 01 | train_macro_f1=0.2066 | val_macro_f1=0.2192

Epoch 2/12
   [train] batch 1/9 | loss=1.0526
   [train] batch 9/9 | loss=0.8097
   [val] batch 1/1 | loss=1.3123
Epoch 02 | train_macro_f1=0.7427 | val_macro_f1=0.2971

Epoch 3/12
   [train] batch 1/9 | loss=0.6380
   [train] batch 9/9 | loss=0.5332
   [val] batch 1/1 | loss=1.1465
Ep

In [1]:
# ============================================================
# FAST RESUME-ONLY STAGE-5 RUNNER
# Skips finished runs automatically using existing metrics.json
# ============================================================

import os
import gc
import json
import time
import copy
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import (
    resnet18, ResNet18_Weights,
    efficientnet_b0, EfficientNet_B0_Weights,
    mobilenet_v3_small, MobileNet_V3_Small_Weights,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)

# ============================================================
# 1. SETTINGS
# ============================================================
SEED = 42
IMG_SIZE = 224
NUM_SAMPLE_FRAMES = 32

# Keep these practical; much larger can slow or OOM.
BATCH_SIZE_BY_MODEL = {
    "resnet18": 24,
    "efficientnet_b0": 16,
    "mobilenet_v3_small": 32,
}

EPOCHS = 12
LR = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 4

USE_AMP = True
USE_PRETRAINED = True
USE_TORCH_COMPILE = True   # if PyTorch 2.x; if unstable, set False

# Faster data loading. If notebook hangs, reduce NUM_WORKERS to 0.
NUM_WORKERS = 4
PIN_MEMORY = True
PERSISTENT_WORKERS = True
PREFETCH_FACTOR = 4

SAMPLE_START_FRAC = 0.10
SAMPLE_END_FRAC = 0.90

FRAME_SOURCES = ["RGB", "RGB_BGREM"]
MODELS_TO_RUN = ["resnet18", "efficientnet_b0", "mobilenet_v3_small"]

# Full protocol list. Finished runs will be skipped automatically.
SPLIT_CONFIGS = [
    ("protocol_A", "protocol_A_train.csv", "protocol_A_val.csv", "protocol_A_test.csv"),

    ("protocol_B_4_to_4", "protocol_B_train_4_test_4.csv", None, "protocol_B_test_4_same.csv"),
    ("protocol_B_4_to_6", "protocol_B_train_4_test_6.csv", None, "protocol_B_test_6.csv"),
    ("protocol_B_4_to_8", "protocol_B_train_4_test_8.csv", None, "protocol_B_test_8.csv"),

    ("protocol_B_6_to_4", "protocol_B_train_6_test_4.csv", None, "protocol_B_test_4.csv"),
    ("protocol_B_6_to_6", "protocol_B_train_6_test_6.csv", None, "protocol_B_test_6_same.csv"),
    ("protocol_B_6_to_8", "protocol_B_train_6_test_8.csv", None, "protocol_B_test_8.csv"),

    ("protocol_B_8_to_4", "protocol_B_train_8_test_4.csv", None, "protocol_B_test_4.csv"),
    ("protocol_B_8_to_6", "protocol_B_train_8_test_6.csv", None, "protocol_B_test_6.csv"),
    ("protocol_B_8_to_8", "protocol_B_train_8_test_8.csv", None, "protocol_B_test_8_same.csv"),

    ("protocol_B_4_6_to_4", "protocol_B_train_4_6_test_4.csv", None, "protocol_B_test_4_from_4_6.csv"),
    ("protocol_B_4_6_to_6", "protocol_B_train_4_6_test_6.csv", None, "protocol_B_test_6_from_4_6.csv"),
    ("protocol_B_4_6_to_8", "protocol_B_train_4_6_test_8.csv", None, "protocol_B_test_8_from_4_6.csv"),

    ("protocol_B_4_8_to_4", "protocol_B_train_4_8_test_4.csv", None, "protocol_B_test_4_from_4_8.csv"),
    ("protocol_B_4_8_to_6", "protocol_B_train_4_8_test_6.csv", None, "protocol_B_test_6_from_4_8.csv"),
    ("protocol_B_4_8_to_8", "protocol_B_train_4_8_test_8.csv", None, "protocol_B_test_8_from_4_8.csv"),

    ("protocol_B_6_8_to_4", "protocol_B_train_6_8_test_4.csv", None, "protocol_B_test_4_from_6_8.csv"),
    ("protocol_B_6_8_to_6", "protocol_B_train_6_8_test_6.csv", None, "protocol_B_test_6_from_6_8.csv"),
    ("protocol_B_6_8_to_8", "protocol_B_train_6_8_test_8.csv", None, "protocol_B_test_8_from_6_8.csv"),

    ("protocol_B_4_6_8_to_4", "protocol_B_train_4_6_8_test_4.csv", None, "protocol_B_test_4_from_4_6_8.csv"),
    ("protocol_B_4_6_8_to_6", "protocol_B_train_4_6_8_test_6.csv", None, "protocol_B_test_6_from_4_6_8.csv"),
    ("protocol_B_4_6_8_to_8", "protocol_B_train_4_6_8_test_8.csv", None, "protocol_B_test_8_from_4_6_8.csv"),
]

CLASS_NAMES = ["doctor", "emergency", "fire", "help", "robbery", "sit_down", "stand_up"]
label2id = {c: i for i, c in enumerate(CLASS_NAMES)}
id2label = {i: c for c, i in label2id.items()}
NUM_CLASSES = len(CLASS_NAMES)

# ============================================================
# 2. REPRODUCIBILITY
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ============================================================
# 3. PATHS
# ============================================================
cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd

SPLIT_DIR = ROOT / "data" / "processed" / "splits"
MANIFEST_DIR = ROOT / "data" / "processed" / "manifests"
FRAME_SUMMARY_CSV = MANIFEST_DIR / "frame_extraction_trimmed_summary.csv"

# IMPORTANT: look for finished runs in ALL prior results roots too
SEARCH_RESULTS_DIRS = [
    ROOT / "results" / "stage5_rgb_master",
    ROOT / "results" / "stage5_remaining_protocols",
    ROOT / "results" / "stage5_resume_all",
]

RESULTS_ROOT = ROOT / "results" / "stage5_resume_all"
CHECKPOINT_ROOT = ROOT / "checkpoints" / "stage5_resume_all"
LOG_ROOT = ROOT / "logs" / "stage5_resume_all"

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
LOG_ROOT.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)

# ============================================================
# 4. DEVICE / SPEED FLAGS
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

# ============================================================
# 5. LOAD FRAME SUMMARY
# ============================================================
frame_df_all = pd.read_csv(FRAME_SUMMARY_CSV)
frame_df_all = frame_df_all[frame_df_all["status"] == "success"].copy()
frame_df_all = frame_df_all[["pair_id", "modality", "output_dir", "frames_extracted"]].drop_duplicates()

# ============================================================
# 6. DISCOVER FINISHED RUNS FROM OLD RESULTS
# ============================================================
finished_keys = set()

for base in SEARCH_RESULTS_DIRS:
    if not base.exists():
        continue
    for mf in base.rglob("metrics.json"):
        try:
            with open(mf, "r") as f:
                m = json.load(f)
            key = (m.get("protocol_name"), m.get("frame_source"), m.get("model_name"))
            if all(key):
                finished_keys.add(key)
        except Exception:
            pass

print("Already finished runs discovered:", len(finished_keys))

# ============================================================
# 7. HELPERS
# ============================================================
def sample_indices_middle_region(n_total, n_samples, start_frac=0.10, end_frac=0.90):
    if n_total <= 0:
        return [0] * n_samples
    start_idx = int(round(start_frac * (n_total - 1)))
    end_idx = int(round(end_frac * (n_total - 1)))
    start_idx = max(0, min(start_idx, n_total - 1))
    end_idx = max(start_idx, min(end_idx, n_total - 1))
    idxs = np.linspace(start_idx, end_idx, n_samples)
    idxs = np.round(idxs).astype(int)
    return np.clip(idxs, 0, n_total - 1).tolist()

def frame_index_to_path(frame_dir, idx):
    return Path(frame_dir) / f"frame_{idx+1:06d}.jpg"

def load_split_df(csv_name):
    csv_path = SPLIT_DIR / csv_name
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing split file: {csv_path}")
    df = pd.read_csv(csv_path)
    if "is_valid_pair" in df.columns:
        df = df[df["is_valid_pair"] == True].copy()
    return df.reset_index(drop=True)

def attach_labels(df_):
    out = df_.copy()
    out = out[out["gesture"].isin(CLASS_NAMES)].copy()
    out["label"] = out["gesture"].map(label2id)
    return out.reset_index(drop=True)

def attach_frame_info(df_, frame_source):
    sub = frame_df_all[frame_df_all["modality"] == frame_source].copy()
    sub = sub.rename(columns={"output_dir": "frame_dir", "frames_extracted": "num_frames_available"})
    sub = sub[["pair_id", "frame_dir", "num_frames_available"]].drop_duplicates()
    out = df_.merge(sub, on="pair_id", how="left")
    out = out[out["num_frames_available"].fillna(0) > 0].copy().reset_index(drop=True)
    out["num_frames_available"] = out["num_frames_available"].astype(int)
    return out

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

class FrameVotingDataset(Dataset):
    def __init__(self, df, transform, num_sample_frames=32):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform
        self.sampled_paths = []
        for _, row in self.df.iterrows():
            idxs = sample_indices_middle_region(
                int(row["num_frames_available"]),
                num_sample_frames,
                SAMPLE_START_FRAC,
                SAMPLE_END_FRAC
            )
            self.sampled_paths.append([frame_index_to_path(row["frame_dir"], i) for i in idxs])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        frames = []
        for p in self.sampled_paths[idx]:
            try:
                with Image.open(p) as f:
                    img = f.convert("RGB")
            except Exception:
                img = Image.fromarray(np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))
            frames.append(self.transform(img))
        frames = torch.stack(frames, dim=0)
        meta = {
            "pair_id": row["pair_id"],
            "subject_id": row["subject_id"],
            "distance": row["distance"],
            "gesture": row["gesture"],
            "frame_dir": row["frame_dir"],
        }
        return frames, int(row["label"]), meta

def collate_fn(batch):
    frames = torch.stack([x[0] for x in batch], dim=0)
    labels = torch.tensor([x[1] for x in batch], dtype=torch.long)
    metas = [x[2] for x in batch]
    return frames, labels, metas

def create_model(model_name):
    if model_name == "resnet18":
        model = resnet18(weights=ResNet18_Weights.DEFAULT if USE_PRETRAINED else None)
        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    elif model_name == "efficientnet_b0":
        model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT if USE_PRETRAINED else None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
    elif model_name == "mobilenet_v3_small":
        model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT if USE_PRETRAINED else None)
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, NUM_CLASSES)
    else:
        raise ValueError(model_name)

    model = model.to(device)
    model = model.to(memory_format=torch.channels_last)

    if USE_TORCH_COMPILE and hasattr(torch, "compile"):
        try:
            model = torch.compile(model, mode="reduce-overhead")
        except Exception:
            pass

    return model

def forward_video_voting(model, frames):
    B, T, C, H, W = frames.shape
    frames = frames.view(B * T, C, H, W).contiguous(memory_format=torch.channels_last)
    logits = model(frames)
    logits = logits.view(B, T, -1)
    return logits.mean(dim=1)

def make_loader(dataset, batch_size, shuffle):
    kwargs = dict(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY and device.type == "cuda",
        collate_fn=collate_fn,
    )
    if NUM_WORKERS > 0:
        kwargs["persistent_workers"] = PERSISTENT_WORKERS
        kwargs["prefetch_factor"] = PREFETCH_FACTOR
    return DataLoader(**kwargs)

def run_one_epoch(model, loader, criterion, optimizer=None, scaler=None, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    all_labels, all_preds = [], []

    for batch_idx, (frames, labels, metas) in enumerate(loader):
        frames = frames.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(USE_AMP and device.type == "cuda")):
                logits = forward_video_voting(model, frames)
                loss = criterion(logits, labels)

            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        preds = logits.argmax(dim=1)
        total_loss += loss.item() * labels.size(0)
        all_labels.extend(labels.detach().cpu().tolist())
        all_preds.extend(preds.detach().cpu().tolist())

        if batch_idx % 10 == 0 or batch_idx == len(loader) - 1:
            mode = "train" if train else "val"
            print(f"   [{mode}] batch {batch_idx+1}/{len(loader)} | loss={loss.item():.4f}")

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(all_labels, all_preds),
        "macro_f1": f1_score(all_labels, all_preds, average="macro"),
        "weighted_f1": f1_score(all_labels, all_preds, average="weighted"),
    }

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_labels, all_preds, rows = [], [], []
    start = time.time()
    total_samples = 0

    for frames, labels, metas in loader:
        frames = frames.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(USE_AMP and device.type == "cuda")):
            logits = forward_video_voting(model, frames)
            probs = torch.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)

        labels_np = labels.detach().cpu().numpy()
        preds_np = preds.detach().cpu().numpy()
        probs_np = probs.detach().cpu().numpy()

        total_samples += len(labels_np)
        all_labels.extend(labels_np.tolist())
        all_preds.extend(preds_np.tolist())

        for i, meta in enumerate(metas):
            row = {
                "pair_id": meta["pair_id"],
                "subject_id": meta["subject_id"],
                "distance": meta["distance"],
                "gesture_true": id2label[int(labels_np[i])],
                "gesture_pred": id2label[int(preds_np[i])],
                "frame_dir": meta["frame_dir"],
            }
            for c_idx, c_name in id2label.items():
                row[f"prob_{c_name}"] = float(probs_np[i][c_idx])
            rows.append(row)

    infer_time = time.time() - start
    videos_per_second = total_samples / infer_time if infer_time > 0 else None
    return all_labels, all_preds, pd.DataFrame(rows), infer_time, videos_per_second

def run_experiment(protocol_name, train_csv_name, val_csv_name, test_csv_name, frame_source, model_name):
    run_name = f"{protocol_name}__{frame_source.lower()}__{model_name}__f{NUM_SAMPLE_FRAMES}__img{IMG_SIZE}"
    results_dir = RESULTS_ROOT / run_name
    checkpoint_dir = CHECKPOINT_ROOT / run_name
    log_dir = LOG_ROOT / run_name
    results_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    log_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "="*100)
    print(f"RUNNING | protocol={protocol_name} | frame_source={frame_source} | model={model_name}")
    print("="*100)

    train_df = load_split_df(train_csv_name)
    test_df = load_split_df(test_csv_name)
    if val_csv_name is None:
        train_df = train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
        n_val = max(1, int(0.10 * len(train_df)))
        val_df = train_df.iloc[:n_val].copy().reset_index(drop=True)
        train_df = train_df.iloc[n_val:].copy().reset_index(drop=True)
    else:
        val_df = load_split_df(val_csv_name)

    train_df = attach_frame_info(attach_labels(train_df), frame_source)
    val_df = attach_frame_info(attach_labels(val_df), frame_source)
    test_df = attach_frame_info(attach_labels(test_df), frame_source)

    print(f"train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

    batch_size = BATCH_SIZE_BY_MODEL[model_name]

    train_loader = make_loader(FrameVotingDataset(train_df, train_transform, NUM_SAMPLE_FRAMES), batch_size, True)
    val_loader = make_loader(FrameVotingDataset(val_df, eval_transform, NUM_SAMPLE_FRAMES), batch_size, False)
    test_loader = make_loader(FrameVotingDataset(test_df, eval_transform, NUM_SAMPLE_FRAMES), batch_size, False)

    model = create_model(model_name)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and device.type == "cuda"))

    best_val_f1 = -1.0
    best_epoch = -1
    best_state_dict = None
    patience_ctr = 0
    history = []

    train_start = time.time()
    for epoch in range(1, EPOCHS + 1):
        print(f"\nEpoch {epoch}/{EPOCHS}")
        train_m = run_one_epoch(model, train_loader, criterion, optimizer, scaler, True)
        val_m = run_one_epoch(model, val_loader, criterion, None, scaler, False)

        history.append({
            "epoch": epoch,
            "train_loss": train_m["loss"],
            "train_acc": train_m["accuracy"],
            "train_macro_f1": train_m["macro_f1"],
            "train_weighted_f1": train_m["weighted_f1"],
            "val_loss": val_m["loss"],
            "val_acc": val_m["accuracy"],
            "val_macro_f1": val_m["macro_f1"],
            "val_weighted_f1": val_m["weighted_f1"],
        })

        if val_m["macro_f1"] > best_val_f1:
            best_val_f1 = val_m["macro_f1"]
            best_epoch = epoch
            patience_ctr = 0
            best_state_dict = copy.deepcopy(model.state_dict())
            torch.save({
                "epoch": epoch,
                "model_state_dict": best_state_dict,
                "protocol_name": protocol_name,
                "frame_source": frame_source,
                "model_name": model_name,
            }, checkpoint_dir / "best_model.pt")
        else:
            patience_ctr += 1

        print(f"Epoch {epoch:02d} | train_macro_f1={train_m['macro_f1']:.4f} | val_macro_f1={val_m['macro_f1']:.4f}")

        if patience_ctr >= PATIENCE:
            print("Early stopping triggered.")
            break

    train_minutes = (time.time() - train_start) / 60.0
    pd.DataFrame(history).to_csv(log_dir / "training_log.csv", index=False)

    model.load_state_dict(best_state_dict)
    test_labels, test_preds, pred_df, infer_seconds, videos_per_second = evaluate(model, test_loader)

    test_accuracy = accuracy_score(test_labels, test_preds)
    test_macro_f1 = f1_score(test_labels, test_preds, average="macro")
    test_weighted_f1 = f1_score(test_labels, test_preds, average="weighted")

    report = classification_report(
        test_labels, test_preds,
        target_names=[id2label[i] for i in range(NUM_CLASSES)],
        output_dict=True, zero_division=0
    )
    report_df = pd.DataFrame(report).transpose()

    pred_csv = results_dir / "test_predictions.csv"
    report_csv = results_dir / "classification_report.csv"
    metrics_json = results_dir / "metrics.json"

    pred_df.to_csv(pred_csv, index=False)
    report_df.to_csv(report_csv)

    per_class_f1 = {}
    for cls in CLASS_NAMES:
        if cls in report_df.index:
            per_class_f1[cls] = float(report_df.loc[cls, "f1-score"])

    metrics = {
        "protocol_name": protocol_name,
        "frame_source": frame_source,
        "model_name": model_name,
        "num_sample_frames": NUM_SAMPLE_FRAMES,
        "batch_size": batch_size,
        "best_epoch": int(best_epoch),
        "best_val_macro_f1": float(best_val_f1),
        "test_accuracy": float(test_accuracy),
        "test_macro_f1": float(test_macro_f1),
        "test_weighted_f1": float(test_weighted_f1),
        "per_class_f1": per_class_f1,
        "train_time_minutes": float(train_minutes),
        "inference_time_seconds": float(infer_seconds),
        "videos_per_second": float(videos_per_second) if videos_per_second is not None else None,
        "train_rows": int(len(train_df)),
        "val_rows": int(len(val_df)),
        "test_rows": int(len(test_df)),
        "predictions_csv": str(pred_csv),
        "classification_report_csv": str(report_csv),
        "checkpoint_best": str(checkpoint_dir / "best_model.pt"),
    }

    with open(metrics_json, "w") as f:
        json.dump(metrics, f, indent=2)

    del model, train_loader, val_loader, test_loader, best_state_dict
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metrics

# ============================================================
# 8. RUN ONLY MISSING
# ============================================================
summary_csv = RESULTS_ROOT / "resume_summary.csv"
all_metrics = []
if summary_csv.exists():
    try:
        all_metrics = pd.read_csv(summary_csv).to_dict(orient="records")
    except Exception:
        all_metrics = []

done_keys = set((m["protocol_name"], m["frame_source"], m["model_name"]) for m in all_metrics if "protocol_name" in m)
done_keys |= finished_keys

pending = []
for frame_source in FRAME_SOURCES:
    for protocol_name, train_csv_name, val_csv_name, test_csv_name in SPLIT_CONFIGS:
        for model_name in MODELS_TO_RUN:
            key = (protocol_name, frame_source, model_name)
            if key not in done_keys:
                pending.append((protocol_name, train_csv_name, val_csv_name, test_csv_name, frame_source, model_name))

print("\nPending runs:", len(pending))
for p in pending[:20]:
    print(" ", (p[0], p[4], p[5]))

for protocol_name, train_csv_name, val_csv_name, test_csv_name, frame_source, model_name in pending:
    metrics = run_experiment(protocol_name, train_csv_name, val_csv_name, test_csv_name, frame_source, model_name)
    all_metrics.append(metrics)
    done_keys.add((protocol_name, frame_source, model_name))
    pd.DataFrame(all_metrics).to_csv(summary_csv, index=False)

print("\nResume finished.")
print("Saved:", summary_csv)

ROOT: /data/Sajjan_Singh/gesture_recognition
RESULTS_ROOT: /data/Sajjan_Singh/gesture_recognition/results/stage5_resume_all
Using device: cuda
GPU name: NVIDIA H100 NVL
Already finished runs discovered: 76

Pending runs: 56
  ('protocol_B_6_to_4', 'RGB', 'resnet18')
  ('protocol_B_6_to_4', 'RGB', 'efficientnet_b0')
  ('protocol_B_6_to_4', 'RGB', 'mobilenet_v3_small')
  ('protocol_B_4_6_8_to_6', 'RGB', 'efficientnet_b0')
  ('protocol_B_4_6_8_to_6', 'RGB', 'mobilenet_v3_small')
  ('protocol_B_4_6_8_to_8', 'RGB', 'resnet18')
  ('protocol_B_4_6_8_to_8', 'RGB', 'efficientnet_b0')
  ('protocol_B_4_6_8_to_8', 'RGB', 'mobilenet_v3_small')
  ('protocol_B_4_to_4', 'RGB_BGREM', 'resnet18')
  ('protocol_B_4_to_4', 'RGB_BGREM', 'efficientnet_b0')
  ('protocol_B_4_to_4', 'RGB_BGREM', 'mobilenet_v3_small')
  ('protocol_B_4_to_6', 'RGB_BGREM', 'resnet18')
  ('protocol_B_4_to_6', 'RGB_BGREM', 'efficientnet_b0')
  ('protocol_B_4_to_6', 'RGB_BGREM', 'mobilenet_v3_small')
  ('protocol_B_6_to_4', 'RGB_BGRE

In [ ]:
# ============================================================
# 05_stage5_vit_convnext_all_protocols.ipynb
# ViT-B/16 + ConvNeXt-Tiny on all protocols
# Resumable, GPU-optimized, stage-5 style runner
# ============================================================

import os
import gc
import json
import time
import copy
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import (
    vit_b_16, ViT_B_16_Weights,
    convnext_tiny, ConvNeXt_Tiny_Weights,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)

# ============================================================
# 1. SETTINGS
# ============================================================
SEED = 42

IMG_SIZE = 224
NUM_SAMPLE_FRAMES = 32

# ViT is heavier; ConvNeXt-Tiny is lighter but still sizable
BATCH_SIZE_BY_MODEL = {
    "vit_b_16": 8,
    "convnext_tiny": 12,
}

EPOCHS = 12
LR = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 4

USE_AMP = True
USE_PRETRAINED = True
USE_TORCH_COMPILE = True

# If dataloader is unstable in notebook, set NUM_WORKERS=0
NUM_WORKERS = 4
PIN_MEMORY = True
PERSISTENT_WORKERS = True
PREFETCH_FACTOR = 4

SAMPLE_START_FRAC = 0.10
SAMPLE_END_FRAC = 0.90

SKIP_IF_METRICS_EXISTS = True

# Choose one or both
FRAME_SOURCES = [
    "RGB",
    "RGB_BGREM",
]

MODELS_TO_RUN = [
    "vit_b_16",
    "convnext_tiny",
]

SPLIT_CONFIGS = [
    ("protocol_A", "protocol_A_train.csv", "protocol_A_val.csv", "protocol_A_test.csv"),

    ("protocol_B_4_to_4", "protocol_B_train_4_test_4.csv", None, "protocol_B_test_4_same.csv"),
    ("protocol_B_4_to_6", "protocol_B_train_4_test_6.csv", None, "protocol_B_test_6.csv"),
    ("protocol_B_4_to_8", "protocol_B_train_4_test_8.csv", None, "protocol_B_test_8.csv"),

    ("protocol_B_6_to_4", "protocol_B_train_6_test_4.csv", None, "protocol_B_test_4.csv"),
    ("protocol_B_6_to_6", "protocol_B_train_6_test_6.csv", None, "protocol_B_test_6_same.csv"),
    ("protocol_B_6_to_8", "protocol_B_train_6_test_8.csv", None, "protocol_B_test_8.csv"),

    ("protocol_B_8_to_4", "protocol_B_train_8_test_4.csv", None, "protocol_B_test_4.csv"),
    ("protocol_B_8_to_6", "protocol_B_train_8_test_6.csv", None, "protocol_B_test_6.csv"),
    ("protocol_B_8_to_8", "protocol_B_train_8_test_8.csv", None, "protocol_B_test_8_same.csv"),

    ("protocol_B_4_6_to_4", "protocol_B_train_4_6_test_4.csv", None, "protocol_B_test_4_from_4_6.csv"),
    ("protocol_B_4_6_to_6", "protocol_B_train_4_6_test_6.csv", None, "protocol_B_test_6_from_4_6.csv"),
    ("protocol_B_4_6_to_8", "protocol_B_train_4_6_test_8.csv", None, "protocol_B_test_8_from_4_6.csv"),

    ("protocol_B_4_8_to_4", "protocol_B_train_4_8_test_4.csv", None, "protocol_B_test_4_from_4_8.csv"),
    ("protocol_B_4_8_to_6", "protocol_B_train_4_8_test_6.csv", None, "protocol_B_test_6_from_4_8.csv"),
    ("protocol_B_4_8_to_8", "protocol_B_train_4_8_test_8.csv", None, "protocol_B_test_8_from_4_8.csv"),

    ("protocol_B_6_8_to_4", "protocol_B_train_6_8_test_4.csv", None, "protocol_B_test_4_from_6_8.csv"),
    ("protocol_B_6_8_to_6", "protocol_B_train_6_8_test_6.csv", None, "protocol_B_test_6_from_6_8.csv"),
    ("protocol_B_6_8_to_8", "protocol_B_train_6_8_test_8.csv", None, "protocol_B_test_8_from_6_8.csv"),

    ("protocol_B_4_6_8_to_4", "protocol_B_train_4_6_8_test_4.csv", None, "protocol_B_test_4_from_4_6_8.csv"),
    ("protocol_B_4_6_8_to_6", "protocol_B_train_4_6_8_test_6.csv", None, "protocol_B_test_6_from_4_6_8.csv"),
    ("protocol_B_4_6_8_to_8", "protocol_B_train_4_6_8_test_8.csv", None, "protocol_B_test_8_from_4_6_8.csv"),
]

CLASS_NAMES = [
    "doctor",
    "emergency",
    "fire",
    "help",
    "robbery",
    "sit_down",
    "stand_up",
]
label2id = {c: i for i, c in enumerate(CLASS_NAMES)}
id2label = {i: c for c, i in label2id.items()}
NUM_CLASSES = len(CLASS_NAMES)

# ============================================================
# 2. REPRODUCIBILITY
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ============================================================
# 3. PATHS
# ============================================================
cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd

SPLIT_DIR = ROOT / "data" / "processed" / "splits"
MANIFEST_DIR = ROOT / "data" / "processed" / "manifests"
FRAME_SUMMARY_CSV = MANIFEST_DIR / "frame_extraction_trimmed_summary.csv"

RESULTS_ROOT = ROOT / "results" / "stage5_vit_convnext"
CHECKPOINT_ROOT = ROOT / "checkpoints" / "stage5_vit_convnext"
LOG_ROOT = ROOT / "logs" / "stage5_vit_convnext"

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
LOG_ROOT.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)

if not FRAME_SUMMARY_CSV.exists():
    raise FileNotFoundError(f"Missing frame summary: {FRAME_SUMMARY_CSV}")

# ============================================================
# 4. DEVICE
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

# ============================================================
# 5. LOAD FRAME SUMMARY
# ============================================================
frame_df_all = pd.read_csv(FRAME_SUMMARY_CSV)
frame_df_all = frame_df_all[frame_df_all["status"] == "success"].copy()
frame_df_all = frame_df_all[["pair_id", "modality", "output_dir", "frames_extracted"]].drop_duplicates()

# ============================================================
# 6. HELPERS
# ============================================================
def sample_indices_middle_region(n_total, n_samples, start_frac=0.10, end_frac=0.90):
    if n_total <= 0:
        return [0] * n_samples

    start_idx = int(round(start_frac * (n_total - 1)))
    end_idx = int(round(end_frac * (n_total - 1)))

    start_idx = max(0, min(start_idx, n_total - 1))
    end_idx = max(start_idx, min(end_idx, n_total - 1))

    idxs = np.linspace(start_idx, end_idx, n_samples)
    idxs = np.round(idxs).astype(int)
    return np.clip(idxs, 0, n_total - 1).tolist()

def frame_index_to_path(frame_dir, idx):
    return Path(frame_dir) / f"frame_{idx+1:06d}.jpg"

def load_split_df(csv_name):
    csv_path = SPLIT_DIR / csv_name
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing split file: {csv_path}")
    df = pd.read_csv(csv_path)
    if "is_valid_pair" in df.columns:
        df = df[df["is_valid_pair"] == True].copy()
    return df.reset_index(drop=True)

def attach_labels(df_):
    out = df_.copy()
    out = out[out["gesture"].isin(CLASS_NAMES)].copy()
    out["label"] = out["gesture"].map(label2id)
    return out.reset_index(drop=True)

def attach_frame_info(df_, frame_source):
    sub = frame_df_all[frame_df_all["modality"] == frame_source].copy()
    sub = sub.rename(columns={
        "output_dir": "frame_dir",
        "frames_extracted": "num_frames_available"
    })
    sub = sub[["pair_id", "frame_dir", "num_frames_available"]].drop_duplicates()

    out = df_.merge(sub, on="pair_id", how="left")
    out = out[out["num_frames_available"].fillna(0) > 0].copy().reset_index(drop=True)
    out["num_frames_available"] = out["num_frames_available"].astype(int)
    return out

# ============================================================
# 7. TRANSFORMS
# ============================================================
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

# ============================================================
# 8. DATASET
# ============================================================
class FrameVotingDataset(Dataset):
    def __init__(self, df, transform, num_sample_frames=32):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform
        self.sampled_paths = []

        print(f"Building sampled frame paths for {len(self.df)} samples...")
        for _, row in self.df.iterrows():
            idxs = sample_indices_middle_region(
                int(row["num_frames_available"]),
                num_sample_frames,
                SAMPLE_START_FRAC,
                SAMPLE_END_FRAC
            )
            self.sampled_paths.append([frame_index_to_path(row["frame_dir"], i) for i in idxs])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        paths = self.sampled_paths[idx]

        frames = []
        for p in paths:
            try:
                with Image.open(p) as f:
                    img = f.convert("RGB")
            except Exception:
                img = Image.fromarray(np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))
            frames.append(self.transform(img))

        frames = torch.stack(frames, dim=0)
        label = int(row["label"])

        meta = {
            "pair_id": row["pair_id"],
            "subject_id": row["subject_id"],
            "distance": row["distance"],
            "gesture": row["gesture"],
            "frame_dir": row["frame_dir"],
        }
        return frames, label, meta

def collate_fn(batch):
    frames = torch.stack([x[0] for x in batch], dim=0)
    labels = torch.tensor([x[1] for x in batch], dtype=torch.long)
    metas = [x[2] for x in batch]
    return frames, labels, metas

def make_loader(dataset, batch_size, shuffle):
    kwargs = dict(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY and device.type == "cuda",
        collate_fn=collate_fn,
    )
    if NUM_WORKERS > 0:
        kwargs["persistent_workers"] = PERSISTENT_WORKERS
        kwargs["prefetch_factor"] = PREFETCH_FACTOR
    return DataLoader(**kwargs)

# ============================================================
# 9. MODEL FACTORY
# ============================================================
def create_model(model_name, num_classes, use_pretrained=True):
    if model_name == "vit_b_16":
        model = vit_b_16(weights=ViT_B_16_Weights.DEFAULT if use_pretrained else None)
        model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)

    elif model_name == "convnext_tiny":
        model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT if use_pretrained else None)
        model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)

    else:
        raise ValueError(f"Unsupported model: {model_name}")

    model = model.to(device)
    model = model.to(memory_format=torch.channels_last)

    if USE_TORCH_COMPILE and hasattr(torch, "compile"):
        try:
            model = torch.compile(model, mode="reduce-overhead")
        except Exception:
            pass

    return model

def forward_video_voting(model, frames):
    B, T, C, H, W = frames.shape
    frames = frames.view(B * T, C, H, W).contiguous(memory_format=torch.channels_last)
    frame_logits = model(frames)
    frame_logits = frame_logits.view(B, T, -1)
    video_logits = frame_logits.mean(dim=1)
    return video_logits

# ============================================================
# 10. TRAIN / EVAL
# ============================================================
def run_one_epoch(model, loader, criterion, optimizer=None, scaler=None, train=True):
    model.train() if train else model.eval()

    total_loss = 0.0
    all_labels = []
    all_preds = []

    total_batches = len(loader)

    for batch_idx, (frames, labels, metas) in enumerate(loader):
        frames = frames.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(USE_AMP and device.type == "cuda")
            ):
                video_logits = forward_video_voting(model, frames)
                loss = criterion(video_logits, labels)

            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        preds = video_logits.argmax(dim=1)

        total_loss += loss.item() * labels.size(0)
        all_labels.extend(labels.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())

        if batch_idx % 10 == 0 or batch_idx == total_batches - 1:
            mode = "train" if train else "val"
            print(f"   [{mode}] batch {batch_idx+1}/{total_batches} | loss={loss.item():.4f}")

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(all_labels, all_preds),
        "macro_f1": f1_score(all_labels, all_preds, average="macro"),
        "weighted_f1": f1_score(all_labels, all_preds, average="weighted"),
    }

@torch.no_grad()
def predict_with_metadata(model, loader):
    model.eval()

    all_labels = []
    all_preds = []
    rows = []

    start_time = time.time()
    total_samples = 0

    for frames, labels, metas in loader:
        frames = frames.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(USE_AMP and device.type == "cuda")
        ):
            video_logits = forward_video_voting(model, frames)
            probs = torch.softmax(video_logits, dim=1)
            preds = probs.argmax(dim=1)

        probs_np = probs.detach().cpu().numpy()
        preds_np = preds.detach().cpu().numpy()
        labels_np = labels.detach().cpu().numpy()

        total_samples += len(labels_np)
        all_labels.extend(labels_np.tolist())
        all_preds.extend(preds_np.tolist())

        for i, meta in enumerate(metas):
            row = {
                "pair_id": meta["pair_id"],
                "subject_id": meta["subject_id"],
                "distance": meta["distance"],
                "gesture_true": id2label[int(labels_np[i])],
                "gesture_pred": id2label[int(preds_np[i])],
                "frame_dir": meta["frame_dir"],
            }
            for c_idx, c_name in id2label.items():
                row[f"prob_{c_name}"] = float(probs_np[i][c_idx])
            rows.append(row)

    elapsed = time.time() - start_time
    videos_per_sec = total_samples / elapsed if elapsed > 0 else None
    return all_labels, all_preds, pd.DataFrame(rows), elapsed, videos_per_sec

# ============================================================
# 11. RUN ONE EXPERIMENT
# ============================================================
def run_experiment(protocol_name, train_csv_name, val_csv_name, test_csv_name, frame_source, model_name):
    print("\n" + "=" * 100)
    print(f"RUNNING | protocol={protocol_name} | frame_source={frame_source} | model={model_name}")
    print("=" * 100)

    train_df = load_split_df(train_csv_name)
    test_df = load_split_df(test_csv_name)

    if val_csv_name is None:
        train_df = train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
        n_val = max(1, int(0.10 * len(train_df)))
        val_df = train_df.iloc[:n_val].copy().reset_index(drop=True)
        train_df = train_df.iloc[n_val:].copy().reset_index(drop=True)
    else:
        val_df = load_split_df(val_csv_name)

    train_df = attach_labels(train_df)
    val_df = attach_labels(val_df)
    test_df = attach_labels(test_df)

    train_df = attach_frame_info(train_df, frame_source)
    val_df = attach_frame_info(val_df, frame_source)
    test_df = attach_frame_info(test_df, frame_source)

    print(f"train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

    batch_size = BATCH_SIZE_BY_MODEL[model_name]

    train_loader = make_loader(
        FrameVotingDataset(train_df, train_transform, NUM_SAMPLE_FRAMES),
        batch_size=batch_size,
        shuffle=True
    )
    val_loader = make_loader(
        FrameVotingDataset(val_df, eval_transform, NUM_SAMPLE_FRAMES),
        batch_size=batch_size,
        shuffle=False
    )
    test_loader = make_loader(
        FrameVotingDataset(test_df, eval_transform, NUM_SAMPLE_FRAMES),
        batch_size=batch_size,
        shuffle=False
    )

    run_name = f"{protocol_name}__{frame_source.lower()}__{model_name}__f{NUM_SAMPLE_FRAMES}__img{IMG_SIZE}"
    results_dir = RESULTS_ROOT / run_name
    checkpoint_dir = CHECKPOINT_ROOT / run_name
    log_dir = LOG_ROOT / run_name

    results_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    log_dir.mkdir(parents=True, exist_ok=True)

    metrics_json = results_dir / "metrics.json"
    if SKIP_IF_METRICS_EXISTS and metrics_json.exists():
        print("Skipping, already finished:", run_name)
        with open(metrics_json, "r") as f:
            return json.load(f)

    model = create_model(model_name, NUM_CLASSES, use_pretrained=USE_PRETRAINED)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and device.type == "cuda"))

    best_val_f1 = -1.0
    best_epoch = -1
    epochs_without_improvement = 0
    history = []

    best_ckpt_path = checkpoint_dir / "best_model.pt"
    best_state_dict = None

    train_start = time.time()

    for epoch in range(1, EPOCHS + 1):
        print(f"\nEpoch {epoch}/{EPOCHS}")

        train_metrics = run_one_epoch(model, train_loader, criterion, optimizer, scaler, train=True)
        val_metrics = run_one_epoch(model, val_loader, criterion, optimizer=None, scaler=scaler, train=False)

        history.append({
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
        })

        print(
            f"Epoch {epoch:02d} | "
            f"train_macro_f1={train_metrics['macro_f1']:.4f} | "
            f"val_macro_f1={val_metrics['macro_f1']:.4f}"
        )

        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1 = val_metrics["macro_f1"]
            best_epoch = epoch
            epochs_without_improvement = 0
            best_state_dict = copy.deepcopy(model.state_dict())
            torch.save({
                "epoch": epoch,
                "model_state_dict": best_state_dict,
                "model_name": model_name,
                "frame_source": frame_source,
                "protocol_name": protocol_name,
                "label2id": label2id,
                "id2label": id2label,
            }, best_ckpt_path)
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            print("Early stopping triggered.")
            break

    train_elapsed = time.time() - train_start
    pd.DataFrame(history).to_csv(log_dir / "training_log.csv", index=False)

    if best_state_dict is None:
        raise RuntimeError("No best model was saved.")

    model.load_state_dict(best_state_dict)

    test_labels, test_preds, pred_df, infer_elapsed, videos_per_sec = predict_with_metadata(model, test_loader)

    test_acc = accuracy_score(test_labels, test_preds)
    test_macro_f1 = f1_score(test_labels, test_preds, average="macro")
    test_weighted_f1 = f1_score(test_labels, test_preds, average="weighted")

    cm = confusion_matrix(test_labels, test_preds)
    report_dict = classification_report(
        test_labels,
        test_preds,
        target_names=[id2label[i] for i in range(NUM_CLASSES)],
        output_dict=True,
        zero_division=0
    )
    report_df = pd.DataFrame(report_dict).transpose()

    pred_csv = results_dir / "test_predictions.csv"
    report_csv = results_dir / "classification_report.csv"
    cm_png = results_dir / "confusion_matrix.png"

    pred_df.to_csv(pred_csv, index=False)
    report_df.to_csv(report_csv)

    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation="nearest")
    plt.title(f"{protocol_name} | {frame_source} | {model_name}")
    plt.colorbar()
    ticks = np.arange(NUM_CLASSES)
    plt.xticks(ticks, [id2label[i] for i in range(NUM_CLASSES)], rotation=45, ha="right")
    plt.yticks(ticks, [id2label[i] for i in range(NUM_CLASSES)])
    plt.xlabel("Predicted")
    plt.ylabel("True")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")
    plt.tight_layout()
    plt.savefig(cm_png, dpi=200, bbox_inches="tight")
    plt.close()

    per_class_f1 = {}
    for cls in CLASS_NAMES:
        if cls in report_df.index:
            per_class_f1[cls] = float(report_df.loc[cls, "f1-score"])

    metrics = {
        "protocol_name": protocol_name,
        "frame_source": frame_source,
        "model_name": model_name,
        "num_sample_frames": NUM_SAMPLE_FRAMES,
        "batch_size": batch_size,
        "best_epoch": int(best_epoch),
        "best_val_macro_f1": float(best_val_f1),
        "test_accuracy": float(test_acc),
        "test_macro_f1": float(test_macro_f1),
        "test_weighted_f1": float(test_weighted_f1),
        "per_class_f1": per_class_f1,
        "train_time_minutes": float(train_elapsed / 60.0),
        "inference_time_seconds": float(infer_elapsed),
        "videos_per_second": float(videos_per_sec) if videos_per_sec is not None else None,
        "train_rows": int(len(train_df)),
        "val_rows": int(len(val_df)),
        "test_rows": int(len(test_df)),
        "predictions_csv": str(pred_csv),
        "classification_report_csv": str(report_csv),
        "confusion_matrix_png": str(cm_png),
        "checkpoint_best": str(best_ckpt_path),
    }

    with open(metrics_json, "w") as f:
        json.dump(metrics, f, indent=4)

    print(json.dumps(metrics, indent=2))

    del model, train_loader, val_loader, test_loader, best_state_dict
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metrics

# ============================================================
# 12. RUN ALL EXPERIMENTS
# ============================================================
summary_csv = RESULTS_ROOT / "stage5_vit_convnext_summary.csv"

all_metrics = []
if summary_csv.exists():
    try:
        prev = pd.read_csv(summary_csv)
        all_metrics = prev.to_dict(orient="records")
    except Exception:
        all_metrics = []

done_keys = set()
for m in all_metrics:
    done_keys.add((m["protocol_name"], m["frame_source"], m["model_name"]))

for frame_source in FRAME_SOURCES:
    for protocol_name, train_csv_name, val_csv_name, test_csv_name in SPLIT_CONFIGS:
        for model_name in MODELS_TO_RUN:
            key = (protocol_name, frame_source, model_name)
            if SKIP_IF_METRICS_EXISTS and key in done_keys:
                print("Already in summary, skipping:", key)
                continue

            metrics = run_experiment(
                protocol_name=protocol_name,
                train_csv_name=train_csv_name,
                val_csv_name=val_csv_name,
                test_csv_name=test_csv_name,
                frame_source=frame_source,
                model_name=model_name
            )
            all_metrics.append(metrics)
            done_keys.add(key)
            pd.DataFrame(all_metrics).to_csv(summary_csv, index=False)

final_df = pd.DataFrame(all_metrics)
final_df.to_csv(summary_csv, index=False)

print("\nDONE.")
print("Saved summary to:", summary_csv)
display(
    final_df[[
        "protocol_name",
        "frame_source",
        "model_name",
        "best_val_macro_f1",
        "test_accuracy",
        "test_macro_f1",
        "test_weighted_f1",
        "train_time_minutes",
        "videos_per_second"
    ]].sort_values(["protocol_name", "frame_source", "test_macro_f1"], ascending=[True, True, False])
)

ROOT: /data/Sajjan_Singh/gesture_recognition
RESULTS_ROOT: /data/Sajjan_Singh/gesture_recognition/results/stage5_vit_convnext
Using device: cuda
GPU name: NVIDIA H100 NVL

RUNNING | protocol=protocol_A | frame_source=RGB | model=vit_b_16
train=525, val=77, test=147
Building sampled frame paths for 525 samples...
Building sampled frame paths for 77 samples...
Building sampled frame paths for 147 samples...


Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /home/sajjan/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth
100%|██████████| 330M/330M [00:04<00:00, 71.9MB/s] 



Epoch 1/12
   [train] batch 1/66 | loss=2.0963
   [train] batch 11/66 | loss=1.7531
   [train] batch 21/66 | loss=1.9482
   [train] batch 31/66 | loss=1.9636
   [train] batch 41/66 | loss=2.0305
   [train] batch 51/66 | loss=1.9194
   [train] batch 61/66 | loss=1.9231
   [train] batch 66/66 | loss=1.9672
   [val] batch 1/10 | loss=1.9976
   [val] batch 10/10 | loss=1.9668
Epoch 01 | train_macro_f1=0.1112 | val_macro_f1=0.0357

Epoch 2/12
   [train] batch 1/66 | loss=1.9410
   [train] batch 11/66 | loss=2.1089
   [train] batch 21/66 | loss=1.9756
   [train] batch 31/66 | loss=2.0322
   [train] batch 41/66 | loss=1.9750
   [train] batch 51/66 | loss=1.9243
   [train] batch 61/66 | loss=1.9894
   [train] batch 66/66 | loss=1.8707
   [val] batch 1/10 | loss=1.8944
   [val] batch 10/10 | loss=1.8609
Epoch 02 | train_macro_f1=0.1237 | val_macro_f1=0.0357

Epoch 3/12
   [train] batch 1/66 | loss=1.9161
   [train] batch 11/66 | loss=1.8944
   [train] batch 21/66 | loss=1.9506
   [train] batch

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /home/sajjan/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth
100%|██████████| 109M/109M [00:01<00:00, 72.2MB/s] 



Epoch 1/12
   [train] batch 1/44 | loss=2.0033
   [train] batch 11/44 | loss=1.9479
   [train] batch 21/44 | loss=1.6495
   [train] batch 31/44 | loss=1.2455
   [train] batch 41/44 | loss=0.8984
   [train] batch 44/44 | loss=0.7895
   [val] batch 1/7 | loss=0.6843
   [val] batch 7/7 | loss=0.5423
Epoch 01 | train_macro_f1=0.3398 | val_macro_f1=0.7203

Epoch 2/12
   [train] batch 1/44 | loss=0.6291
   [train] batch 11/44 | loss=0.4694
   [train] batch 21/44 | loss=0.4239
   [train] batch 31/44 | loss=0.2873
   [train] batch 41/44 | loss=0.3066
   [train] batch 44/44 | loss=0.3534
   [val] batch 1/7 | loss=0.1940
   [val] batch 7/7 | loss=0.3632
Epoch 02 | train_macro_f1=0.8138 | val_macro_f1=0.8533

Epoch 3/12
   [train] batch 1/44 | loss=0.2332
   [train] batch 11/44 | loss=0.1210
   [train] batch 21/44 | loss=0.1058
   [train] batch 31/44 | loss=0.1412
   [train] batch 41/44 | loss=0.2613
   [train] batch 44/44 | loss=0.0367
   [val] batch 1/7 | loss=0.1843
   [val] batch 7/7 | loss=

In [1]:
# ============================================================
# OPTIMIZED RESUMABLE RUNNER — ViT-B/16 + ConvNeXt-Tiny
# Fixes: OOM from torch.compile CUDA graphs, proper batch sizes
# Tuned for NVIDIA H100 NVL with other processes sharing GPU
# ============================================================

import os, gc, json, time, copy, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import (
    vit_b_16, ViT_B_16_Weights,
    convnext_tiny, ConvNeXt_Tiny_Weights,
)

warnings.filterwarnings("ignore")

# ── tell PyTorch allocator to avoid fragmentation ───────────
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# ============================================================
# 1. SETTINGS
# ============================================================
SEED              = 42
IMG_SIZE          = 224
NUM_SAMPLE_FRAMES = 32

# Safe batch sizes given ~15 GB free on a shared H100
# (B × T frames go through model at once: B*32 forward passes)
BATCH_SIZE_BY_MODEL = {
    "vit_b_16":      16,   # 16 videos × 32 frames = 512 forward passes per step
    "convnext_tiny": 24,   # 24 × 32 = 768 forward passes per step
}

EPOCHS       = 12
LR           = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE     = 4

USE_AMP           = True
USE_PRETRAINED    = True
USE_TORCH_COMPILE = False   # DISABLED — causes 64 GB CUDA graph pre-allocation = OOM

# Gradient accumulation: effectively doubles batch without extra VRAM
# Set to 2 → effective batch = 2× above values
GRAD_ACCUM_STEPS = 2

# DataLoader
NUM_WORKERS        = 8    # safe default; raise if CPU is not bottleneck
PIN_MEMORY         = True
PERSISTENT_WORKERS = True
PREFETCH_FACTOR    = 2

SAMPLE_START_FRAC = 0.10
SAMPLE_END_FRAC   = 0.90
SKIP_IF_METRICS_EXISTS = True

FRAME_SOURCES = ["RGB", "RGB_BGREM"]
MODELS_TO_RUN = ["vit_b_16", "convnext_tiny"]

SPLIT_CONFIGS = [
    ("protocol_A",            "protocol_A_train.csv",            "protocol_A_val.csv",  "protocol_A_test.csv"),
    ("protocol_B_4_to_4",     "protocol_B_train_4_test_4.csv",   None, "protocol_B_test_4_same.csv"),
    ("protocol_B_4_to_6",     "protocol_B_train_4_test_6.csv",   None, "protocol_B_test_6.csv"),
    ("protocol_B_4_to_8",     "protocol_B_train_4_test_8.csv",   None, "protocol_B_test_8.csv"),
    ("protocol_B_6_to_4",     "protocol_B_train_6_test_4.csv",   None, "protocol_B_test_4.csv"),
    ("protocol_B_6_to_6",     "protocol_B_train_6_test_6.csv",   None, "protocol_B_test_6_same.csv"),
    ("protocol_B_6_to_8",     "protocol_B_train_6_test_8.csv",   None, "protocol_B_test_8.csv"),
    ("protocol_B_8_to_4",     "protocol_B_train_8_test_4.csv",   None, "protocol_B_test_4.csv"),
    ("protocol_B_8_to_6",     "protocol_B_train_8_test_6.csv",   None, "protocol_B_test_6.csv"),
    ("protocol_B_8_to_8",     "protocol_B_train_8_test_8.csv",   None, "protocol_B_test_8_same.csv"),
    ("protocol_B_4_6_to_4",   "protocol_B_train_4_6_test_4.csv", None, "protocol_B_test_4_from_4_6.csv"),
    ("protocol_B_4_6_to_6",   "protocol_B_train_4_6_test_6.csv", None, "protocol_B_test_6_from_4_6.csv"),
    ("protocol_B_4_6_to_8",   "protocol_B_train_4_6_test_8.csv", None, "protocol_B_test_8_from_4_6.csv"),
    ("protocol_B_4_8_to_4",   "protocol_B_train_4_8_test_4.csv", None, "protocol_B_test_4_from_4_8.csv"),
    ("protocol_B_4_8_to_6",   "protocol_B_train_4_8_test_6.csv", None, "protocol_B_test_6_from_4_8.csv"),
    ("protocol_B_4_8_to_8",   "protocol_B_train_4_8_test_8.csv", None, "protocol_B_test_8_from_4_8.csv"),
    ("protocol_B_6_8_to_4",   "protocol_B_train_6_8_test_4.csv", None, "protocol_B_test_4_from_6_8.csv"),
    ("protocol_B_6_8_to_6",   "protocol_B_train_6_8_test_6.csv", None, "protocol_B_test_6_from_6_8.csv"),
    ("protocol_B_6_8_to_8",   "protocol_B_train_6_8_test_8.csv", None, "protocol_B_test_8_from_6_8.csv"),
    ("protocol_B_4_6_8_to_4", "protocol_B_train_4_6_8_test_4.csv", None, "protocol_B_test_4_from_4_6_8.csv"),
    ("protocol_B_4_6_8_to_6", "protocol_B_train_4_6_8_test_6.csv", None, "protocol_B_test_6_from_4_6_8.csv"),
    ("protocol_B_4_6_8_to_8", "protocol_B_train_4_6_8_test_8.csv", None, "protocol_B_test_8_from_4_6_8.csv"),
]

CLASS_NAMES = ["doctor", "emergency", "fire", "help", "robbery", "sit_down", "stand_up"]
label2id    = {c: i for i, c in enumerate(CLASS_NAMES)}
id2label    = {i: c for c, i in label2id.items()}
NUM_CLASSES = len(CLASS_NAMES)

# ============================================================
# 2. SEED & DEVICE
# ============================================================
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    free, total = torch.cuda.mem_get_info(0)
    print(f"GPU : {props.name}")
    print(f"VRAM: {total/1e9:.1f} GB total | {free/1e9:.1f} GB free right now")

torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

# ============================================================
# 3. PATHS
# ============================================================
cwd  = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd

SPLIT_DIR         = ROOT / "data" / "processed" / "splits"
MANIFEST_DIR      = ROOT / "data" / "processed" / "manifests"
FRAME_SUMMARY_CSV = MANIFEST_DIR / "frame_extraction_trimmed_summary.csv"

RESULTS_ROOT    = ROOT / "results"     / "stage5_vit_convnext"
CHECKPOINT_ROOT = ROOT / "checkpoints" / "stage5_vit_convnext"
LOG_ROOT        = ROOT / "logs"        / "stage5_vit_convnext"

for d in [RESULTS_ROOT, CHECKPOINT_ROOT, LOG_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
if not FRAME_SUMMARY_CSV.exists():
    raise FileNotFoundError(f"Missing: {FRAME_SUMMARY_CSV}")

# ============================================================
# 4. FRAME SUMMARY
# ============================================================
frame_df_all = pd.read_csv(FRAME_SUMMARY_CSV)
frame_df_all = frame_df_all[frame_df_all["status"] == "success"].copy()
frame_df_all = (frame_df_all[["pair_id", "modality", "output_dir", "frames_extracted"]]
                .drop_duplicates())

# ============================================================
# 5. HELPERS
# ============================================================
def sample_indices_middle_region(n_total, n_samples, s=0.10, e=0.90):
    if n_total <= 0:
        return [0] * n_samples
    si = max(0, min(int(round(s * (n_total - 1))), n_total - 1))
    ei = max(si, min(int(round(e * (n_total - 1))), n_total - 1))
    return np.clip(np.round(np.linspace(si, ei, n_samples)).astype(int), 0, n_total - 1).tolist()

def frame_index_to_path(frame_dir, idx):
    return Path(frame_dir) / f"frame_{idx+1:06d}.jpg"

def load_split_df(csv_name):
    p = SPLIT_DIR / csv_name
    if not p.exists():
        raise FileNotFoundError(f"Missing split: {p}")
    df = pd.read_csv(p)
    if "is_valid_pair" in df.columns:
        df = df[df["is_valid_pair"] == True].copy()
    return df.reset_index(drop=True)

def attach_labels(df_):
    out = df_[df_["gesture"].isin(CLASS_NAMES)].copy()
    out["label"] = out["gesture"].map(label2id)
    return out.reset_index(drop=True)

def attach_frame_info(df_, frame_source):
    sub = (frame_df_all[frame_df_all["modality"] == frame_source]
           .rename(columns={"output_dir": "frame_dir",
                            "frames_extracted": "num_frames_available"})
           [["pair_id", "frame_dir", "num_frames_available"]]
           .drop_duplicates())
    out = df_.merge(sub, on="pair_id", how="left")
    out = out[out["num_frames_available"].fillna(0) > 0].copy().reset_index(drop=True)
    out["num_frames_available"] = out["num_frames_available"].astype(int)
    return out

# ============================================================
# 6. TRANSFORMS
# ============================================================
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# ============================================================
# 7. DATASET
# ============================================================
class FrameVotingDataset(Dataset):
    def __init__(self, df, transform, n_frames=32):
        self.df        = df.reset_index(drop=True).copy()
        self.transform = transform
        self.paths     = []
        print(f"  Building frame paths for {len(self.df)} samples...")
        for _, row in self.df.iterrows():
            idxs = sample_indices_middle_region(
                int(row["num_frames_available"]), n_frames,
                SAMPLE_START_FRAC, SAMPLE_END_FRAC)
            self.paths.append([frame_index_to_path(row["frame_dir"], i) for i in idxs])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        blank = torch.zeros(3, IMG_SIZE, IMG_SIZE)
        frames = []
        for p in self.paths[idx]:
            try:
                with Image.open(p) as f:
                    frames.append(self.transform(f.convert("RGB")))
            except Exception:
                frames.append(blank)
        return (
            torch.stack(frames),
            int(row["label"]),
            {"pair_id":    row["pair_id"],
             "subject_id": row["subject_id"],
             "distance":   row["distance"],
             "gesture":    row["gesture"],
             "frame_dir":  row["frame_dir"]},
        )

def collate_fn(batch):
    return (
        torch.stack([x[0] for x in batch]),
        torch.tensor([x[1] for x in batch], dtype=torch.long),
        [x[2] for x in batch],
    )

def make_loader(ds, batch_size, shuffle):
    kw = dict(dataset=ds, batch_size=batch_size, shuffle=shuffle,
              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
              collate_fn=collate_fn, drop_last=False)
    if NUM_WORKERS > 0:
        kw["persistent_workers"] = PERSISTENT_WORKERS
        kw["prefetch_factor"]    = PREFETCH_FACTOR
    return DataLoader(**kw)

# ============================================================
# 8. MODEL — no torch.compile (avoids CUDA-graph OOM)
# ============================================================
def create_model(model_name, num_classes, pretrained=True):
    if model_name == "vit_b_16":
        m = vit_b_16(weights=ViT_B_16_Weights.DEFAULT if pretrained else None)
        m.heads.head = nn.Linear(m.heads.head.in_features, num_classes)
    elif model_name == "convnext_tiny":
        m = convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None)
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, num_classes)
    else:
        raise ValueError(model_name)

    # channels_last speeds up ConvNeXt significantly on H100
    m = m.to(device).to(memory_format=torch.channels_last)
    return m

def forward_voting(model, frames):
    """Flatten B×T frames into one GPU call, then mean-pool logits."""
    B, T, C, H, W = frames.shape
    flat   = frames.view(B * T, C, H, W).contiguous(memory_format=torch.channels_last)
    logits = model(flat)              # (B*T, num_classes)
    return logits.view(B, T, -1).mean(1)   # (B, num_classes)

# ============================================================
# 9. ADAPTIVE BATCH SIZE — auto-reduce on OOM
# ============================================================
def safe_forward(model, frames, criterion, labels):
    """Try forward pass; if OOM, halve batch and retry (up to 3 times)."""
    for attempt in range(3):
        try:
            with torch.autocast("cuda", dtype=torch.float16,
                                enabled=(USE_AMP and device.type == "cuda")):
                logits = forward_voting(model, frames)
                loss   = criterion(logits, labels)
            return logits, loss, frames.shape[0]
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            half = frames.shape[0] // 2
            if half == 0:
                raise
            print(f"  ⚠ OOM — retrying with batch={half}")
            frames = frames[:half]
            labels = labels[:half]
    raise RuntimeError("Repeated OOM — reduce BATCH_SIZE_BY_MODEL manually.")

# ============================================================
# 10. TRAIN / EVAL LOOPS
# ============================================================
def run_epoch(model, loader, criterion, optimizer=None, scaler=None, train=True):
    model.train() if train else model.eval()
    total_loss, n_samples = 0.0, 0
    labels_all, preds_all = [], []
    n_batches = len(loader)

    accum_count = 0
    if train:
        optimizer.zero_grad(set_to_none=True)

    t_batch = time.time()

    for bi, (frames, labels, _) in enumerate(loader):
        frames = frames.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.set_grad_enabled(train):
            if train:
                with torch.autocast("cuda", dtype=torch.float16,
                                    enabled=(USE_AMP and device.type == "cuda")):
                    logits = forward_voting(model, frames)
                    loss   = criterion(logits, labels) / GRAD_ACCUM_STEPS
                scaler.scale(loss).backward()
                accum_count += 1
                if accum_count == GRAD_ACCUM_STEPS or bi == n_batches - 1:
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    accum_count = 0
            else:
                with torch.autocast("cuda", dtype=torch.float16,
                                    enabled=(USE_AMP and device.type == "cuda")):
                    logits = forward_voting(model, frames)
                    loss   = criterion(logits, labels)

        bs_actual = labels.size(0)
        total_loss += loss.item() * (GRAD_ACCUM_STEPS if train else 1) * bs_actual
        n_samples  += bs_actual
        preds = logits.detach().argmax(1)
        labels_all.extend(labels.cpu().numpy())
        preds_all.extend(preds.cpu().numpy())

        # print every 10 batches
        if bi % 10 == 0 or bi == n_batches - 1:
            elapsed = time.time() - t_batch
            vps     = n_samples / elapsed if elapsed > 0 else 0
            mode    = "train" if train else "val"
            print(f"  [{mode}] {bi+1:4d}/{n_batches} | "
                  f"loss={loss.item()*(GRAD_ACCUM_STEPS if train else 1):.4f} | "
                  f"{vps:.1f} vid/s", flush=True)

    return {
        "loss":        total_loss / max(n_samples, 1),
        "accuracy":    accuracy_score(labels_all, preds_all),
        "macro_f1":    f1_score(labels_all, preds_all, average="macro"),
        "weighted_f1": f1_score(labels_all, preds_all, average="weighted"),
    }

@torch.no_grad()
def predict(model, loader):
    model.eval()
    labels_all, preds_all, rows = [], [], []
    t0 = time.time()
    for frames, labels, metas in loader:
        frames = frames.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16,
                            enabled=(USE_AMP and device.type == "cuda")):
            logits = forward_voting(model, frames)
            probs  = torch.softmax(logits, 1)
            preds  = probs.argmax(1)
        probs_np  = probs.cpu().numpy()
        preds_np  = preds.cpu().numpy()
        labels_np = labels.numpy()
        labels_all.extend(labels_np)
        preds_all.extend(preds_np)
        for i, meta in enumerate(metas):
            r = {"pair_id":      meta["pair_id"],
                 "subject_id":   meta["subject_id"],
                 "distance":     meta["distance"],
                 "gesture_true": id2label[int(labels_np[i])],
                 "gesture_pred": id2label[int(preds_np[i])],
                 "frame_dir":    meta["frame_dir"]}
            for ci, cn in id2label.items():
                r[f"prob_{cn}"] = float(probs_np[i][ci])
            rows.append(r)
    elapsed = time.time() - t0
    return labels_all, preds_all, pd.DataFrame(rows), elapsed, len(labels_all) / elapsed

# ============================================================
# 11. CHECKPOINT RESUME
# ============================================================
def load_checkpoint_if_exists(best_ckpt, model, optimizer, scaler):
    """
    Returns (start_epoch, best_val_f1, history).
    Looks for training_state.pt first (full resume),
    then falls back to best_model.pt (weights-only resume).
    """
    state_path = best_ckpt.parent / "training_state.pt"
    if state_path.exists():
        print(f"  ✔ Resuming full state from {state_path.name}")
        s = torch.load(state_path, map_location=device)
        model.load_state_dict(s["model_state_dict"])
        optimizer.load_state_dict(s["optimizer_state_dict"])
        scaler.load_state_dict(s["scaler_state_dict"])
        return s["epoch"] + 1, s["best_val_f1"], s.get("history", [])
    if best_ckpt.exists():
        print(f"  ✔ Found best_model.pt — loading weights, fresh optimizer")
        s = torch.load(best_ckpt, map_location=device)
        model.load_state_dict(s["model_state_dict"])
        return 1, -1.0, []
    print("  Starting fresh from epoch 1")
    return 1, -1.0, []

def save_training_state(best_ckpt, model, optimizer, scaler, epoch, best_val_f1, history):
    torch.save({
        "epoch":                epoch,
        "best_val_f1":          best_val_f1,
        "history":              history,
        "model_state_dict":     model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict":    scaler.state_dict(),
    }, best_ckpt.parent / "training_state.pt")

# ============================================================
# 12. SINGLE EXPERIMENT
# ============================================================
def run_experiment(protocol_name, train_csv, val_csv, test_csv, frame_source, model_name):
    print("\n" + "=" * 100)
    print(f"  protocol={protocol_name} | source={frame_source} | model={model_name}")
    print("=" * 100)

    run_name = (f"{protocol_name}__{frame_source.lower()}__{model_name}"
                f"__f{NUM_SAMPLE_FRAMES}__img{IMG_SIZE}")
    results_dir    = RESULTS_ROOT    / run_name
    checkpoint_dir = CHECKPOINT_ROOT / run_name
    log_dir        = LOG_ROOT        / run_name
    for d in [results_dir, checkpoint_dir, log_dir]:
        d.mkdir(parents=True, exist_ok=True)

    metrics_json = results_dir / "metrics.json"
    if SKIP_IF_METRICS_EXISTS and metrics_json.exists():
        print("  Already complete — skipping.")
        with open(metrics_json) as f:
            return json.load(f)

    # ── data ────────────────────────────────────────────────
    train_df = load_split_df(train_csv)
    test_df  = load_split_df(test_csv)
    if val_csv is None:
        train_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
        n_val    = max(1, int(0.10 * len(train_df)))
        val_df   = train_df.iloc[:n_val].copy().reset_index(drop=True)
        train_df = train_df.iloc[n_val:].copy().reset_index(drop=True)
    else:
        val_df = load_split_df(val_csv)

    train_df = attach_frame_info(attach_labels(train_df), frame_source)
    val_df   = attach_frame_info(attach_labels(val_df),   frame_source)
    test_df  = attach_frame_info(attach_labels(test_df),  frame_source)
    print(f"  train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

    bs           = BATCH_SIZE_BY_MODEL[model_name]
    train_loader = make_loader(
        FrameVotingDataset(train_df, train_transform, NUM_SAMPLE_FRAMES), bs, True)
    val_loader   = make_loader(
        FrameVotingDataset(val_df,   eval_transform,  NUM_SAMPLE_FRAMES), bs, False)
    test_loader  = make_loader(
        FrameVotingDataset(test_df,  eval_transform,  NUM_SAMPLE_FRAMES), bs, False)

    # ── model + optimiser ────────────────────────────────────
    model     = create_model(model_name, NUM_CLASSES, USE_PRETRAINED)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler    = torch.amp.GradScaler("cuda", enabled=(USE_AMP and device.type == "cuda"))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS, eta_min=1e-6)

    best_ckpt = checkpoint_dir / "best_model.pt"

    # ── resume ───────────────────────────────────────────────
    start_epoch, best_val_f1, history = load_checkpoint_if_exists(
        best_ckpt, model, optimizer, scaler)

    best_state_dict       = None
    best_epoch            = max(start_epoch - 1, 0)
    epochs_no_improvement = 0

    if history:
        # reconstruct patience counter from saved history
        epochs_no_improvement = 0
        for h in reversed(history):
            if h["val_macro_f1"] < best_val_f1:
                epochs_no_improvement += 1
            else:
                break
        print(f"  Resuming epoch {start_epoch} | best_f1={best_val_f1:.4f} | "
              f"patience={epochs_no_improvement}/{PATIENCE}")

    # advance scheduler to match already-done epochs
    for _ in range(start_epoch - 1):
        scheduler.step()

    # ── training ─────────────────────────────────────────────
    train_start = time.time()

    for epoch in range(start_epoch, EPOCHS + 1):
        lr_now = scheduler.get_last_lr()[0] if epoch > 1 else LR
        print(f"\nEpoch {epoch}/{EPOCHS}  lr={lr_now:.2e}")

        tr = run_epoch(model, train_loader, criterion, optimizer, scaler, train=True)
        va = run_epoch(model, val_loader,   criterion, train=False)
        scheduler.step()

        row = {
            "epoch":            epoch,
            "train_loss":       tr["loss"],
            "train_acc":        tr["accuracy"],
            "train_macro_f1":   tr["macro_f1"],
            "train_weighted_f1":tr["weighted_f1"],
            "val_loss":         va["loss"],
            "val_acc":          va["accuracy"],
            "val_macro_f1":     va["macro_f1"],
            "val_weighted_f1":  va["weighted_f1"],
        }
        history.append(row)
        print(f"  → train_f1={tr['macro_f1']:.4f} | val_f1={va['macro_f1']:.4f}")

        if va["macro_f1"] > best_val_f1:
            best_val_f1           = va["macro_f1"]
            best_epoch            = epoch
            epochs_no_improvement = 0
            best_state_dict       = copy.deepcopy(model.state_dict())
            torch.save({"epoch": epoch, "model_state_dict": best_state_dict,
                        "model_name": model_name, "frame_source": frame_source,
                        "protocol_name": protocol_name,
                        "label2id": label2id, "id2label": id2label}, best_ckpt)
        else:
            epochs_no_improvement += 1

        # save full state every epoch for true resume
        save_training_state(best_ckpt, model, optimizer, scaler,
                            epoch, best_val_f1, history)
        pd.DataFrame(history).to_csv(log_dir / "training_log.csv", index=False)

        if epochs_no_improvement >= PATIENCE:
            print("  Early stopping.")
            break

    train_elapsed = time.time() - train_start

    # ── evaluation ───────────────────────────────────────────
    if best_state_dict is None:
        if best_ckpt.exists():
            best_state_dict = torch.load(best_ckpt, map_location=device)["model_state_dict"]
        else:
            raise RuntimeError("No checkpoint found.")
    model.load_state_dict(best_state_dict)

    test_labels, test_preds, pred_df, infer_t, vps = predict(model, test_loader)

    acc    = accuracy_score(test_labels, test_preds)
    mf1    = f1_score(test_labels, test_preds, average="macro")
    wf1    = f1_score(test_labels, test_preds, average="weighted")
    cm     = confusion_matrix(test_labels, test_preds)
    rep    = classification_report(test_labels, test_preds,
                                   target_names=[id2label[i] for i in range(NUM_CLASSES)],
                                   output_dict=True, zero_division=0)
    rep_df = pd.DataFrame(rep).transpose()

    pred_csv = results_dir / "test_predictions.csv"
    rep_csv  = results_dir / "classification_report.csv"
    cm_png   = results_dir / "confusion_matrix.png"

    pred_df.to_csv(pred_csv, index=False)
    rep_df.to_csv(rep_csv)

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, interpolation="nearest")
    ax.set_title(f"{protocol_name} | {frame_source} | {model_name}")
    plt.colorbar(im, ax=ax)
    ticks = np.arange(NUM_CLASSES)
    ax.set_xticks(ticks); ax.set_yticks(ticks)
    ax.set_xticklabels([id2label[i] for i in range(NUM_CLASSES)], rotation=45, ha="right")
    ax.set_yticklabels([id2label[i] for i in range(NUM_CLASSES)])
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=8)
    plt.tight_layout()
    plt.savefig(cm_png, dpi=150, bbox_inches="tight")
    plt.close()

    per_cls_f1 = {cn: float(rep_df.loc[cn, "f1-score"])
                  for cn in CLASS_NAMES if cn in rep_df.index}

    metrics = {
        "protocol_name":          protocol_name,
        "frame_source":           frame_source,
        "model_name":             model_name,
        "num_sample_frames":      NUM_SAMPLE_FRAMES,
        "batch_size":             bs,
        "effective_batch_size":   bs * GRAD_ACCUM_STEPS,
        "best_epoch":             int(best_epoch),
        "best_val_macro_f1":      float(best_val_f1),
        "test_accuracy":          float(acc),
        "test_macro_f1":          float(mf1),
        "test_weighted_f1":       float(wf1),
        "per_class_f1":           per_cls_f1,
        "train_time_minutes":     float(train_elapsed / 60),
        "inference_time_seconds": float(infer_t),
        "videos_per_second":      float(vps) if vps else None,
        "train_rows":             len(train_df),
        "val_rows":               len(val_df),
        "test_rows":              len(test_df),
        "predictions_csv":        str(pred_csv),
        "classification_report_csv": str(rep_csv),
        "confusion_matrix_png":   str(cm_png),
        "checkpoint_best":        str(best_ckpt),
    }

    with open(metrics_json, "w") as f:
        json.dump(metrics, f, indent=4)
    print(json.dumps({k: v for k, v in metrics.items()
                      if k not in ("per_class_f1",)}, indent=2))

    # ── free VRAM before next experiment ─────────────────────
    del model, train_loader, val_loader, test_loader, best_state_dict
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

    return metrics

# ============================================================
# 13. RUN ALL (resume-aware)
# ============================================================
summary_csv = RESULTS_ROOT / "stage5_vit_convnext_summary.csv"

all_metrics: list = []
if summary_csv.exists():
    try:
        all_metrics = pd.read_csv(summary_csv).to_dict(orient="records")
    except Exception:
        pass

done_keys = {(m["protocol_name"], m["frame_source"], m["model_name"])
             for m in all_metrics}

print(f"\nAlready done: {len(done_keys)} experiments")

for frame_source in FRAME_SOURCES:
    for protocol_name, train_csv, val_csv, test_csv in SPLIT_CONFIGS:
        for model_name in MODELS_TO_RUN:
            key = (protocol_name, frame_source, model_name)
            if SKIP_IF_METRICS_EXISTS and key in done_keys:
                print(f"  Skip (done): {key}")
                continue
            m = run_experiment(
                protocol_name, train_csv, val_csv, test_csv,
                frame_source, model_name)
            all_metrics.append(m)
            done_keys.add(key)
            pd.DataFrame(all_metrics).to_csv(summary_csv, index=False)

final_df = pd.DataFrame(all_metrics)
final_df.to_csv(summary_csv, index=False)

print("\n ALL DONE — summary:", summary_csv)
print(final_df[[
    "protocol_name", "frame_source", "model_name",
    "best_val_macro_f1", "test_accuracy", "test_macro_f1",
    "train_time_minutes", "videos_per_second",
]].sort_values(["protocol_name", "frame_source", "test_macro_f1"],
               ascending=[True, True, False]).to_string())

Device: cuda
GPU : NVIDIA H100 NVL
VRAM: 99.9 GB total | 90.6 GB free right now
ROOT: /data/Sajjan_Singh/gesture_recognition

Already done: 31 experiments
  Skip (done): ('protocol_A', 'RGB', 'vit_b_16')
  Skip (done): ('protocol_A', 'RGB', 'convnext_tiny')
  Skip (done): ('protocol_B_4_to_4', 'RGB', 'vit_b_16')
  Skip (done): ('protocol_B_4_to_4', 'RGB', 'convnext_tiny')
  Skip (done): ('protocol_B_4_to_6', 'RGB', 'vit_b_16')
  Skip (done): ('protocol_B_4_to_6', 'RGB', 'convnext_tiny')
  Skip (done): ('protocol_B_4_to_8', 'RGB', 'vit_b_16')
  Skip (done): ('protocol_B_4_to_8', 'RGB', 'convnext_tiny')
  Skip (done): ('protocol_B_6_to_4', 'RGB', 'vit_b_16')
  Skip (done): ('protocol_B_6_to_4', 'RGB', 'convnext_tiny')
  Skip (done): ('protocol_B_6_to_6', 'RGB', 'vit_b_16')
  Skip (done): ('protocol_B_6_to_6', 'RGB', 'convnext_tiny')
  Skip (done): ('protocol_B_6_to_8', 'RGB', 'vit_b_16')
  Skip (done): ('protocol_B_6_to_8', 'RGB', 'convnext_tiny')
  Skip (done): ('protocol_B_8_to_4', 'RG